In [ ]:
# Import the os module for environment-variable access
import os

# Select the "conservative" Gold calibration profile for this run
os.environ["ROGII_GOLD_PROFILE"] = "conservative"

In [ ]:
# Import standard-library modules used for environment setup
import sys, os, glob, subprocess

# koolbox setup: install from a bundled wheel, or fall back to sys.path
# Default offline location where the koolbox package is expected to live
kb_dir = '/kaggle/input/koolbox-offline'

# If the default directory is missing, search the input tree for it
if not os.path.isdir(kb_dir):
    # Look anywhere under /kaggle/input for a folder starting with "koolbox"
    cand = glob.glob('/kaggle/input/**/koolbox*', recursive=True)
    # Show the first few candidates found for debugging
    print('koolbox candidates:', cand[:5])
    # Adopt the first candidate as the koolbox directory if any were found
    if cand: kb_dir = cand[0]

# Report which directory will be used as the koolbox source
print('using koolbox dir:', kb_dir)

# Proceed only if the resolved koolbox directory actually exists
if os.path.isdir(kb_dir):
    # List a sample of the directory contents for debugging
    print('listing:', os.listdir(kb_dir)[:20])
    # Collect every wheel file found anywhere under the koolbox directory
    whls = glob.glob(f'{kb_dir}/**/*.whl', recursive=True)
    # Prefer installing wheels when they are present
    if whls:
        # Install each discovered wheel without pulling extra dependencies
        for w in whls:
            # Announce the wheel about to be installed
            print('install', w)
            # Run pip to install this wheel, ignoring failures
            subprocess.run(['pip', 'install', '--no-deps', w], check=False)
    # No wheels found, so make the package importable directly from disk
    else:
        # Put the koolbox directory at the front of the import search path
        sys.path.insert(0, kb_dir)
        # also try subdirs so a nested package layout still imports
        for sub in os.listdir(kb_dir):
            # Build the absolute path to this sub-entry
            sub_path = os.path.join(kb_dir, sub)
            # Add the sub-directory to the import path when it is a folder
            if os.path.isdir(sub_path):
                sys.path.insert(0, sub_path)

# Import the now-available koolbox package
import koolbox

# Confirm the import succeeded and show where it was loaded from
print('koolbox OK:', koolbox.__file__)

In [ ]:
# Gradient-boosting regressor plus its logging and early-stopping callbacks
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
# Root-mean-squared-error scoring function used as the competition metric
from sklearn.metrics import root_mean_squared_error
# Group-aware K-fold splitter so rows from one well stay in a single fold
from sklearn.model_selection import GroupKFold
# Ridge linear model used for stacking/blending
from sklearn.linear_model import Ridge
# CatBoost gradient-boosting regressor, an alternative base learner
from catboost import CatBoostRegressor
# KD-tree for fast nearest-neighbour spatial lookups
from scipy.spatial import cKDTree
# Savitzky-Golay filter for smoothing noisy log curves
from scipy.signal import savgol_filter
# Helpers for running work in parallel across CPU cores
from joblib import Parallel, delayed
# Project-specific training utility
from koolbox import Trainer
# Object-oriented filesystem path handling
from pathlib import Path
# Just-in-time compiler decorator for speeding up numeric loops
from numba import njit
# Plotting library
import matplotlib.pyplot as plt
# Standard library for querying CPU count, etc.
import multiprocessing
# Statistical visualisation built on matplotlib
import seaborn as sns
# Tabular data handling
import pandas as pd
# Numerical array computing
import numpy as np
# Standard warnings control
import warnings
# Model/object serialisation
import joblib
# Timing utilities
import time
# Filename globbing
import glob
# Operating-system interfaces
import os

# Silence non-critical warnings to keep notebook output readable
warnings.filterwarnings("ignore")

In [ ]:
# Central configuration object holding paths, seeds, and the CV scheme
class CFG:
    # Root directory of the competition dataset
    dataset_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    # Directory holding precomputed artifacts (cached features, models)
    artifacts_path = Path("/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts")

    # Global random seed for reproducibility
    seed = 42
    # Number of cross-validation folds
    n_splits = 5
    # Group-aware cross-validation splitter keyed on well id
    cv = GroupKFold(n_splits=n_splits)

    # Scoring function used throughout training and evaluation
    metric = root_mean_squared_error

In [ ]:
# Row-count threshold that splits wells into "small" vs "large" eval bins
SELECTOR_N_EVAL_THRESHOLD = 4840.0
# Two cut points on vertical (Z) span used to bucket wells into three bands
SELECTOR_Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)

# Maps each well bin code to the prediction variant that works best for it
SELECTOR_BIN_VARIANTS = {
    0: 'pf_scale_5_hold_0.2',
    1: 'pf_scale_3_hold_0.15',
    2: 'pf_scale_12_beam_0.2_hold_0.15',
    3: 'pf_scale_5_hold_0.15',
    4: 'pf_scale_5_beam_0.05_hold_0.05',
    5: 'pf_scale_12_beam_0.2_hold_0.05',
}

# Fallback variant used when a bin code has no specific mapping
SELECTOR_GLOBAL_VARIANT = 'pf_scale_8_hold_0.2'
# Likelihood-tempering scales evaluated when ensembling the particle filter
SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)

# Geological formation/contact column names present in the type-well data
FORMATION_COLS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

# Beam-search hyper-parameter sets: (beam_size, move_cost, gr_err_scale, smooth_radius)
BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2),
    (10,  8.0,  64.0, 2),
    ( 8, 35.0, 220.0, 1),
    (10, 14.0,  90.0, 5),
    (20,  4.0,  36.0, 3),
    (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
    (20, 30.0, 200.0, 2),
    (15, 10.0,  80.0, 4),
    (25,  6.0,  50.0, 3),
    (10, 40.0, 300.0, 1),
    (12, 18.0, 120.0, 5),
    (30,  8.0,  70.0, 2),
    (10, 50.0, 400.0, 0),
]

# Derive a horizontal-well TVT curve from formation contacts in the type well
def tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
    # Keep only type-well rows that carry a geology label
    tw_g = tw_tr.dropna(subset=['Geology'])
    # Shallowest TVT at which the reference formation appears
    ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    # If the reference formation is absent, fall back to the first available one
    if np.isnan(ref_tvt):
        # Use the first geology label present as the new reference
        ref_col = tw_g['Geology'].iloc[0]
        # Recompute the reference TVT for that fallback formation
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    # Mean residual that aligns the contact-derived curve to known TVT values
    offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
    # Return the offset-corrected TVT estimate along the horizontal well
    return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset

# Load the horizontal-well and type-well CSVs for a given well id
def load_well(wid, split='train'):
    # Directory for the requested split (train/test)
    base = CFG.dataset_path / split
    # Horizontal-well measurements
    hw = pd.read_csv(base / f'{wid}__horizontal_well.csv')
    # Reference type-well measurements
    tw = pd.read_csv(base / f'{wid}__typewell.csv')
    # Hand both tables back to the caller
    return hw, tw

# Estimate horizontal-well TVT by tracking GR against the type well with a particle filter
def run_particle_filter(hw, tw, n_particles=500, seed=42):
    # Sort the type well by TVT so it can be used as a lookup table
    tw_s   = tw.sort_values('TVT')
    # TVT axis of the type-well reference curve
    tw_tvt = tw_s['TVT'].values.astype(float)
    # GR values of the reference curve, gaps filled with the mean
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    # Rows where TVT is already known (the conditioning prefix)
    kn = hw[hw['TVT_input'].notna()]
    # Rows to be predicted (TVT unknown)
    ev = hw[hw['TVT_input'].isna()]
    # Nothing to predict -> return the inputs unchanged with zero log-likelihood
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0

    # Last known row anchors the filter's starting state
    last     = kn.iloc[-1]
    # Its TVT, depth (Z) and measured depth (MD)
    last_tvt = float(last['TVT_input'])
    last_Z   = float(last['Z'])
    last_MD  = float(last['MD'])

    # Expected type-well GR at each known TVT
    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    # GR measurement noise, estimated from known rows and clipped to a sane range
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))

    # Use the last 30 known rows to estimate the dip/rate trend
    tail = kn.tail(30)
    # Step-wise changes in TVT, Z and MD over that tail
    dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values)
    # Only use steps with positive MD advance
    m  = dm > 0
    # Median initial rate of (TVT+Z) change per unit MD, or 0 if too few steps
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    # Number of particles
    N   = n_particles
    # Seeded random generator for reproducibility
    rng = np.random.default_rng(seed)
    # Latent "stratigraphic level" (TVT + Z) at the last known point
    ls   = last_tvt + last_Z
    # sp45 patch (sel15 vb best)
    pos  = ls + 4.5 * rng.standard_normal(N)
    # Per-particle rate, jittered around the estimated initial rate
    rate = ir + 0.01 * rng.standard_normal(N)
    # Uniform initial particle weights
    w    = np.ones(N) / N

    # Filter constants: momentum, process/observation noise, jitter, resample threshold
    MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5

    # MD and Z arrays for the rows to be predicted
    md_v = ev['MD'].values.astype(float)
    z_v  = ev['Z'].values.astype(float)
    # Interpolate GR gaps before tracking
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    # GR observations aligned to the eval rows
    gr_v = gr_interp.values.astype(float)[ev.index]

    # Output buffer seeded with the known inputs
    out_vals = hw['TVT_input'].values.astype(float).copy()
    # Per-eval-row predicted TVT
    res = np.empty(len(ev))
    # Track the previous MD to compute step lengths
    prev_MD = last_MD
    # Accumulated model evidence (log-likelihood)
    log_lik = 0.0

    # Walk forward through every eval row, advancing the particle cloud
    for i in range(len(ev)):
        # MD advanced since the previous row, floored at 1 to avoid zero steps
        dm_step = max(md_v[i] - prev_MD, 1.0)
        # Random-walk the rate with momentum plus small noise
        rate = MOM * rate + VN * rng.standard_normal(N)
        # Advance latent position by rate*step plus process noise
        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
        # Convert latent level to TVT by removing the row's depth
        tvt_p = pos - z_v[i]
        # Keep TVT within the type-well range (plus margin)
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        # Re-fold the clipped TVT back into latent position
        pos   = tvt_p + z_v[i]

        # Expected GR at each particle's TVT
        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        # Standardised GR residual per particle
        d  = (gr_v[i] - eg) / gs
        # Gaussian likelihood, with the exponent capped for numerical safety
        lk = np.exp(-0.5 * np.minimum(d**2, 600.))
        # Floor the likelihood away from exact zero
        lk = np.maximum(lk, 1e-300)
        # Weighted average likelihood across particles (for evidence)
        avg_lk = float((w * lk).sum())
        # Accumulate log evidence
        log_lik += np.log(max(avg_lk, 1e-300))
        # Reweight particles by their likelihood
        w = w * lk
        # Normalising constant
        ws = w.sum()
        # Renormalise, or reset to uniform if all weights collapsed
        w = w / ws if ws > 0 else np.ones(N) / N

        # Effective sample size of the weighted cloud
        n_eff = 1.0 / (w**2).sum()
        # Resample (systematic) when the cloud degenerates
        if n_eff < RESAMP * N:
            # Cumulative weight distribution
            cum = np.cumsum(w)
            # Single uniform offset for systematic resampling
            u0  = rng.uniform(0, 1.0 / N)
            # Resample indices at evenly-spaced positions
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            # Replace positions and add roughening jitter
            pos  = pos[idx]  + RP * rng.standard_normal(N)
            # Replace rates and add roughening jitter
            rate = rate[idx] + RR * rng.standard_normal(N)
            # Reset weights to uniform after resampling
            w    = np.ones(N) / N

        # Posterior-mean TVT estimate for this row
        res[i] = float(np.dot(w, pos - z_v[i]))
        # Advance the previous-MD marker
        prev_MD = md_v[i]

    # Write the predicted TVTs back into the eval rows
    out_vals[list(ev.index)] = res
    # Return the completed TVT curve and the model evidence
    return out_vals, log_lik

# Likelihood-weighted ensemble of particle-filter runs across many seeds
def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
    # Per-seed predictions
    preds = []
    # Per-seed log-likelihoods
    liks  = []
    # Run the filter once per seed
    for s in range(n_seeds):
        # Single filter run
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        # Collect its prediction
        preds.append(p)
        # Collect its evidence
        liks.append(ll)

    # Stack evidences into an array
    liks   = np.array(liks)
    # Shift by the max for numerical stability before exponentiating
    liks_n = liks - liks.max()
    # Temper the evidences into ensemble weights
    weights = np.exp(liks_n / scale)
    # Normalise the weights to sum to one
    weights /= weights.sum()

    # Return the weighted average prediction across seeds
    return (weights[:, None] * np.stack(preds, 0)).sum(0)

# Same likelihood ensemble but returning a prediction for each tempering scale
def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128):
    # Per-seed predictions
    preds = []
    # Per-seed log-likelihoods
    liks = []
    # Run the filter once per seed
    for s in range(n_seeds):
        # Single filter run
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        # Collect its prediction
        preds.append(p)
        # Collect its evidence
        liks.append(ll)
    # Stack all seed predictions into one array
    pred_arr = np.stack(preds, 0)
    # Stack evidences
    liks = np.array(liks)
    # Stabilise before exponentiating
    liks_n = liks - liks.max()
    # Output dict keyed by scale name
    out = {}
    # Build one weighted-average prediction per scale
    for scale in scales:
        # Temper evidences at this scale
        weights = np.exp(liks_n / float(scale))
        # Normalise weights
        weights /= weights.sum()
        # Store the weighted-average prediction under its scale key
        out[f'pf_scale_{scale:g}'] = (weights[:, None] * pred_arr).sum(0)
    # Also store the plain (unweighted) seed mean
    out['pf_mean'] = pred_arr.mean(0)
    # Return all scale variants
    return out

# Dynamic-programming beam search matching horizontal GR to the type-well curve
def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
    # Number of horizontal GR samples to place
    n  = len(hgr)
    # Length of the type-well reference curve
    nt = len(tw_tvt)
    # Degenerate case: nothing to place
    if n == 0:
        return np.array([last_tvt])

    # Optionally Savitzky-Golay smooth the horizontal GR when enough points exist
    if r > 0 and n > max(3, 2 * r + 1):
        # Odd smoothing window bounded by the data length
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
        # Smoothed GR signal
        sgr = savgol_filter(hgr, win, min(2, win - 1))
    else:
        # No smoothing: use the raw signal
        sgr = hgr.copy()

    # Index on the reference curve closest to the last known TVT (start state)
    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))

    # Allowed index moves at each step
    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
    # Movement cost penalising larger jumps
    MC    = mc * np.array([2., 1., 0., 1., 2.])

    # Beam state indices, all starting at the start index
    bidx  = np.full(bs, si, dtype=np.int64)
    # Beam path costs, all infinite except the seed
    bcost = np.full(bs, np.inf)
    # Seed beam has zero cost
    bcost[0] = 0.
    # Current number of active beams
    bn = 1

    # Output TVT per horizontal sample
    result = np.zeros(n)

    # Advance the beam one horizontal sample at a time
    for step in range(n):
        # Current (smoothed) GR value
        gv = sgr[step]
        # Candidate next indices for every active beam and move
        ni = bidx[:bn, None] + MOVES[None, :]
        # Clipped indices for safe lookup
        ci = np.clip(ni, 0, nt - 1)
        # Mask of in-range candidate moves
        valid = (ni >= 0) & (ni < nt)

        # GR mismatch cost at each candidate
        gr_e = (gv - tw_gr[ci])**2 / es
        # Total cost = running cost + GR cost + move cost
        tot  = bcost[:bn, None] + gr_e + MC[None, :]
        # Invalidate out-of-range candidates
        tot  = np.where(valid, tot, np.inf)

        # Flatten candidates for selection
        ni_f  = ni.flatten()
        tot_f = tot.flatten()
        vf    = valid.flatten()
        # Drop invalid candidates
        ni_f  = ni_f[vf]
        tot_f = tot_f[vf]

        # Sort surviving candidates by cost
        order = np.argsort(tot_f)
        ni_s  = ni_f[order]
        tot_s = tot_f[order]

        # Keep only the cheapest occurrence of each distinct index
        _, first = np.unique(ni_s, return_index=True)
        ni_u  = ni_s[first]
        tot_u = tot_s[first]

        # How many beams we can keep this step
        kept = min(bs, len(ni_u))
        # Partial-select the cheapest "kept" candidates
        top  = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
        # Order those kept candidates by cost
        top  = top[np.argsort(tot_u[top])]

        # Update beam indices with the survivors
        bidx[:kept]  = ni_u[top]
        # Update beam costs with the survivors
        bcost[:kept] = tot_u[top]
        # Pad any unused beam slots
        if kept < bs:
            # Repeat the last survivor's index into empty slots
            bidx[kept:]  = bidx[kept - 1]
            # Mark padded slots with infinite cost
            bcost[kept:] = np.inf
        # Record the active beam count
        bn = kept

        # Best beam's TVT becomes this step's output
        result[step] = tw_tvt[bidx[0]]

    # Return the full TVT path
    return result

# Average several beam-search configurations into one TVT prediction
def run_beam_ensemble(hw, tw):
    # Known (conditioning) rows
    kn = hw[hw['TVT_input'].notna()]
    # Rows to predict
    ev = hw[hw['TVT_input'].isna()]
    # Nothing to predict -> return inputs unchanged
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()

    # Last known TVT used as the beam start
    last_tvt = float(kn.iloc[-1]['TVT_input'])
    # Type well sorted by TVT
    tw_s  = tw.sort_values('TVT')
    # Reference TVT axis
    tw_tvt = tw_s['TVT'].values.astype(float)
    # Reference GR curve with gaps filled
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    # Full horizontal GR with gaps interpolated
    gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    # GR restricted to the eval rows
    hgr    = gr_all[ev.index]

    # Run beam search once per configuration
    beam_results = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
                    for (bs, mc, es, r) in BEAM_CONFIGS]

    # Average the configurations together
    beam_mean = np.stack(beam_results, 0).mean(0)

    # Write the averaged beam result into the eval rows
    out = hw['TVT_input'].values.astype(float).copy()
    out[list(ev.index)] = beam_mean
    # Return the completed curve
    return out

# Classify a well into a bin code and pick its prediction variant
def selector_well_code(hw):
    # Boolean mask of rows needing prediction
    eval_mask = hw['TVT_input'].isna().to_numpy()
    # Count of eval rows
    n_eval = float(eval_mask.sum())
    # Depths of the eval rows
    z_eval = hw.loc[eval_mask, 'Z'].values.astype(float)
    # Vertical span covered by the eval section
    z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
    # Binary bin from the eval-row count threshold
    n_bin = int(n_eval > SELECTOR_N_EVAL_THRESHOLD)
    # Three-way bin from the Z-span thresholds
    z_bin = int(np.searchsorted(SELECTOR_Z_SPAN_THRESHOLDS, z_span, side='right'))
    # Combine into a single bin code
    code = n_bin + 2 * z_bin
    # Look up the variant for this code, falling back to the global one
    variant = SELECTOR_BIN_VARIANTS.get(code, SELECTOR_GLOBAL_VARIANT)
    # Return the code, variant name, and the features used
    return code, variant, n_eval, z_span

# Parse a variant name like "pf_scale_5_beam_0.2_hold_0.15" into numbers
def parse_selector_variant(name):
    # Split the name on underscores
    parts = name.split('_')
    # Third token is always the PF tempering scale
    scale = float(parts[2])
    # Default beam mixing weight
    beam_weight = 0.0
    # Default hold (last-known) mixing weight
    hold_weight = 0.0
    # Read the beam weight if the name encodes one
    if 'beam' in parts:
        beam_weight = float(parts[parts.index('beam') + 1])
    # Read the hold weight if the name encodes one
    if 'hold' in parts:
        hold_weight = float(parts[parts.index('hold') + 1])
    # Return the three parsed mixing parameters
    return scale, beam_weight, hold_weight

# Build a final prediction by blending PF, beam, and last-known per the variant
def apply_selector_variant(name, pf_by_scale, tvt_beam, last_known_tvt):
    # Parse the mixing weights out of the variant name
    scale, beam_weight, hold_weight = parse_selector_variant(name)
    # Base particle-filter prediction at the requested scale
    base = pf_by_scale.get(f'pf_scale_{scale:g}')
    # Fall back to the global variant's base if that scale is missing
    if base is None:
        base = pf_by_scale[SELECTOR_GLOBAL_VARIANT.split('_beam_')[0].split('_hold_')[0]]
    # Blend the base PF prediction with the beam prediction
    pred = (1.0 - beam_weight) * base + beam_weight * tvt_beam
    # Blend the result toward the last-known TVT (the "hold" term)
    pred = (1.0 - hold_weight) * pred + hold_weight * last_known_tvt
    # Return the final blended prediction
    return pred

In [ ]:
# Global random seed
SEED=42
# Cap worker threads at 4 (or fewer if the machine has fewer cores)
NCPU=min(4,multiprocessing.cpu_count())

# Geological formation/contact columns used as spatial anchors
FORMATIONS=["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
# Plane-KNN neighbour count; dense imputer samples-per-well, neighbours; CV folds
PLANE_K=10; DENSE_SPW=60; DENSE_K=20; N_SPLITS=5

# Beam-search configs: (beam_size, move_cost, gr_err_scale, smooth_radius, tag)
BEAMS=[
    (10,20.0,144.0,2,"cons"),
    (10, 8.0, 64.0,2,"loose"),
    ( 8,35.0,220.0,1,"vcons"),
    (10,14.0, 90.0,5,"sm5"),
    (20, 4.0, 36.0,3,"vloose"),
    (12,12.0,100.0,3,"mid"),
    (15,25.0,180.0,2,"stiff"),
]

# Particle counts for the z-tracking filter and the ANCC filter
PF_N=600; ANCC_N=600
# Z-filter dynamics: velocity momentum, velocity noise, position noise
PF_MOM=0.993; PF_VN=0.005; PF_PN=0.01
# GR-noise sigma bounds and default
PF_GR_SIG_MIN=10.; PF_GR_SIG_MAX=60.; PF_GR_SIG_DEF=30.
# Z-filter init velocity std, init spread, resample threshold
PF_INIT_V_STD=0.02; PF_INIT_SPR=0.5; PF_RESAMP=0.5
# Roughening noise (pos/vel), smoothing window, smoothed-GR weight
PF_ROUGH_P=0.2; PF_ROUGH_V=0.003; PF_GR_WIN=5; PF_GR_WT=0.3
# ANCC-filter dynamics: rate momentum, rate noise, position noise
ANCC_ALPHA=0.998; ANCC_RN=0.002; ANCC_PN=0.005
# ANCC-filter init rate, init spread, roughening (pos/rate)
ANCC_IR=0.01; ANCC_IS=0.3; ANCC_RP=0.1; ANCC_RR=0.001

# JIT: linear interpolation on a uniform grid (clamped at the ends)
@njit(cache=True)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i]*(1.-t) + grid[i+1]*t

# JIT: systematic resampling of particle (position, aux) pairs with roughening
@njit(cache=True)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N+1)
    for j in range(N): cum[j+1]=cum[j]+w[j]
    u0=np.random.uniform(0.,1./N)
    np2=np.empty(N); na=np.empty(N); ci=0
    for j in range(N):
        u=u0+j/N
        while ci<N-1 and cum[ci+1]<u: ci+=1
        np2[j]=pos[ci]+rp*np.random.randn()
        na[j] =aux[ci]+rv*np.random.randn()
    return np2,na

# JIT: dynamic-programming beam search over +/-2 index moves against the type-well GR.
# Tracks beam history (hI/hP) to backtrack the lowest-cost path at the end.
@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    """Beam search Ã‚Â±2 delta, Numba JIT."""
    n=len(sgr); nt=len(tw_gr); MAX=BS*6
    bidx=np.zeros(BS,np.int64); bidx[0]=si
    bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
    # Forward pass: expand each beam, dedupe candidates, keep the cheapest BS
    for step in range(n):
        gv=sgr[step]; nc=np.int64(0)
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(-2,3):            # +/-2: TVT can go down
                ni=idx+d
                if ni<0 or ni>=nt: continue
                tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                fnd=np.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        kept=min(BS,nc)
        # Selection sort the kept-best candidates to the front
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]
                cC[i],cC[mi]=cC[mi],cC[i]
                cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    # Pick the cheapest terminal beam, then backtrack its index path
    best=np.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=np.zeros(n,np.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path

# JIT: ANCC particle filter tracking latent level (pos) with a drifting rate.
# Returns posterior-mean TVT (pts) and its std per row (std_).
@njit(cache=True)
def _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,
              ALPHA,RN,PN,IS,RP,RR,RESAMP):
    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ls+IS*np.random.randn()
        rate[j]=ir+0.01*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
    # March forward one row at a time: predict, weight by GR, resample, estimate
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        for j in range(N):
            rate[j]=ALPHA*rate[j]+RN*np.random.randn()
            pos[j]+=rate[j]*dm+PN*np.random.randn()
            tvt_j=pos[j]-z_v[i]
            tvt_j=max(tvt_j,vmin-50.); tvt_j=min(tvt_j,vmin+len(gg)*step+50.)
            pos[j]=tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                d=(gr_v[i]-eg)/gs
                lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,rate=_resamp(pos,rate,w,N,RP,RR)
            for j in range(N): w[j]=1./N
        tv=0.
        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
        pts[i]=tv; va=0.
        for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i]=va**0.5; pm=md_v[i]
    return pts,std_

# JIT: Z-aware particle filter. Tracks TVT directly with a velocity model and an
# extra likelihood term tying velocity to the dz/dMD trend (beta,icpt,zsig).
@njit(cache=True)
def _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,
          gs,ip,iv,beta,icpt,zsig,N,
          MOM,VN,PN,GR_WT,RP,RV,RESAMP):
    pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ip+0.5*np.random.randn()
        vel[j]=iv+0.02*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
    # Forward pass with combined raw-GR, smoothed-GR, and velocity likelihoods
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
        for j in range(N):
            vel[j]=MOM*vel[j]+VN*np.random.randn()
            pos[j]+=vel[j]*dm+PN*np.random.randn()
            pos[j]=max(pos[j],vmin-50.); pos[j]=min(pos[j],vmin+len(gg_p)*step+50.)
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                ep=_interp1(gg_p,pos[j],vmin,step)
                dp=(gr_v[i]-ep)/gs
                lp=max(np.exp(-0.5*dp*dp) if dp*dp<600. else 0.,1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es=_interp1(gg_s,pos[j],vmin,step)
                    ds=(gr_sm_v[i]-es)/(gs*1.5)
                    ls=max(np.exp(-0.5*ds*ds) if ds*ds<600. else 0.,1e-300)
                    lk=(1.-GR_WT)*lp+GR_WT*ls
                else: lk=lp
                lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ws2=0.
        for j in range(N):
            dv=(vel[j]-ve)/max(zsig*2.,0.005)
            lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
            w[j]*=lz; ws2+=w[j]
        if ws2>0.:
            for j in range(N): w[j]/=ws2
        else:
            for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,vel=_resamp(pos,vel,w,N,RP,RV)
            for j in range(N): w[j]=1./N
        wm=0.
        for j in range(N): wm+=w[j]*pos[j]
        pts[i]=wm; va=0.
        for j in range(N): va+=w[j]*(pos[j]-wm)**2
        std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
    return pts,std_

# Dense grid for O(1) typewell lookup
def _grid(tw_tvt,tw_gr,step=0.2):
    # Build a uniform TVT grid spanning the type well and interpolate GR onto it
    tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
    tvt_g=np.arange(tmin,tmax+step,step)
    return np.interp(tvt_g,tw_tvt,tw_gr).astype(np.float64),float(tmin),float(step)

# Estimate GR observation noise from the known prefix (clipped to bounds)
def _gr_sig(hw,tw_tvt,tw_gr):
    kn=hw[hw['TVT_input'].notna()&hw['GR'].notna()]
    if len(kn)<20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn['GR'].values-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)),
                          PF_GR_SIG_MIN,PF_GR_SIG_MAX))

# Nearest-index lookup into a sorted array
def _nn(arr,v):
    i=int(np.searchsorted(arr,v,'left'))
    if i>=len(arr): return len(arr)-1
    if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
    return i

# Interpolate gaps then optionally rolling-mean smooth a 1-D signal
def _smooth(vals,fb,r):
    s=pd.Series(vals,dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)

# Run one beam-search config and map the index path back to TVT values
def beam_search(gr_h,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):
    si=_nn(tw_tvt,start_tvt)
    sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)
    path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))
    return tw_tvt[path].astype(np.float32)

# Prepare inputs and run the ANCC particle filter for a well
def run_pf_ancc(hw,tw_tvt,tw_gr,N=ANCC_N):
    gs=_gr_sig(hw,tw_tvt,tw_gr)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
    tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
    dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr)
    pts,std=_pf_ancc(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                      ev['GR'].values.astype(np.float64),gg,gmin,gst,
                      gs,ls,ir,N,ANCC_ALPHA,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,PF_RESAMP)
    return pts.astype(np.float32),std.astype(np.float32)

# Prepare inputs (including the dz/dMD velocity regression) and run the Z filter
def run_pf_z(hw,tw_tvt,tw_gr,N=PF_N):
    gs=_gr_sig(hw,tw_tvt,tw_gr)
    tw_s=pd.Series(tw_gr).rolling(PF_GR_WIN,center=True,min_periods=1).mean().values.astype(np.float32)
    kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)
    dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0
    if m2.sum()>=10:
        vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]
        A=np.column_stack([vz,np.ones_like(vz)]); c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
        beta,icpt,zsig=float(c[0]),float(c[1]),max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
    else: beta,icpt,zsig=-1.,0.,0.1
    t2=kna.tail(20); dvt2=np.diff(t2['TVT_input'].values); dmd2=np.diff(t2['MD'].values); m3=dmd2>0
    iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr)
    gs2,_,_=_grid(tw_tvt,tw_s)
    gr_sm=hw['GR'].rolling(PF_GR_WIN,center=True,min_periods=1).mean()
    pts,std=_pf_z(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                   ev['GR'].values.astype(np.float64),
                   gr_sm.loc[ev.index].values.astype(np.float64),
                   gg,gs2,gmin,gst,gs,float(kna['TVT_input'].iloc[-1]),iv,
                   beta,icpt,zsig,N,
                   PF_MOM,PF_VN,PF_PN,PF_GR_WT,PF_ROUGH_P,PF_ROUGH_V,PF_RESAMP)
    return pts.astype(np.float32),std.astype(np.float32)

# Warm up / compile the JIT kernels once on tiny dummy inputs
_md=np.linspace(1,50,20,np.float64); _z=np.zeros(20,np.float64); _gr=np.full(20,50.,np.float64)
_gg=np.linspace(45,55,100,np.float64)
_pf_ancc(_md,_z,_gr,_gg,45.,0.1,20.,50.,0.,8,0.998,0.002,0.005,0.3,0.1,0.001,0.5)
_pf_z(_md,_z,_gr,_gr,_gg,_gg,45.,0.1,20.,50.,0.,-1.,0.,0.1,8,0.993,0.005,0.01,0.3,0.2,0.003,0.5)
_beam_jit(np.random.randn(30),np.random.randn(50),25,8,15.,100.)

# Outlier-robust linear slope of y vs x (returns 0 when ill-conditioned)
def robust_slope(x,y,w=None):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<2 or np.std(x[m])<1e-6: return 0.
    return float(np.polyfit(x[m],y[m],1)[0])

# Fit an affine GR calibration (slope a, offset b) of known GR vs type-well GR
def affine_cal(kgr,tw_at_k,min_pts=20):
    v=np.isfinite(kgr)&np.isfinite(tw_at_k)
    if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
        return 1.,float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
    a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)

# Segmented stratigraphic offset b_well across prefix thirds plus a tail-weighted fit
def seg_b_well(ktvt,kz,form_col):
    """Segment b_well: early/mid/late thirds + full prefix.
    Returns (b_full, b_early, b_mid, b_late, b_wls) for feature richness."""
    bv=ktvt+kz-form_col; n=len(bv)
    b_full=float(np.median(bv))
    b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
    t1,t2=n//3, 2*n//3
    b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
    b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
    # WLS (tail-upweighted)
    w=np.exp(0.02*np.arange(n)); w/=w.sum()
    b_wls=float(np.dot(w,bv))
    return b_full,b_early,b_mid,b_late,b_wls

# Multi-scale normalized cross-correlation between known and horizontal GR
def multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
    """Multi-scale NCC. Returns score-weighted ensemble + per-scale signals."""
    out=[]
    # Correlate at each window half-width, recording best-match TVT and its score
    for hw in hws:
        win=2*hw+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
        if M==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw,mode='edge')
        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))
    # Score-weighted ensemble (NEW: softmax-weighted combination)
    tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
    sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
    sc_ens=(tvts*sw).sum(1).astype(np.float32)
    # Return per-scale (tvt,score) pairs plus the softmax ensemble
    return out, sc_ens   # [(tvt8,sc8),(tvt15,sc15),(tvt25,sc25)], ensemble

# Spatial plane-fit imputer: predicts each formation from XY of nearby wells
class FormationPlaneKNN:
    # Build per-well median XY and formation depths, indexed by a KD-tree
    def __init__(self,well_ids,data_dir):
        rows=[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
            except: continue
            if len(df)==0: continue
            row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
            rows.append(row)
        self.df=pd.DataFrame(rows); self.wmap={w:i for i,w in enumerate(self.df['wid'])}
        xy=self.df[['x','y']].to_numpy(); self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
        self.tree=cKDTree(xy/self.scale)
        self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
        self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

    # Fit a local weighted plane (in XY) through k neighbours and evaluate at query points
    def impute(self,xy_q,self_wid=None,k=PLANE_K):
        q=xy_q/self.scale; nf=min(k+5,len(self.df))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
        xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]; wx=w*xn; wy=w*yn
        A=np.zeros((len(q),3,3))
        A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
        A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
        A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
        A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
        rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
        try: coef=np.linalg.solve(A,rhs)
        except:
            coef=np.zeros((len(q),3,6))
            for r in range(len(q)):
                try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                except: pass
        Xq=xy_q[:,0]; Yq=xy_q[:,1]
        pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
        pred[~vk.any(1)]=self.fa.mean(0)
        return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)

# Dense ANCC imputer: inverse-distance interpolation over sampled points from all wells
class DenseANCCImputer:
    # Subsample each well's (X,Y,ANCC) points and index them with a KD-tree
    def __init__(self,well_ids,data_dir,spw=DENSE_SPW):
        xs,ys,anccs,wids=[],[],[],[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
            except: continue
            if len(df)==0: continue
            ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int); s=df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values)
            anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))
        self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
        self.ancc=np.concatenate(anccs).astype(np.float32); self.wids=np.array(wids)
        self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
        self.tree=cKDTree(self.xy/self.scale)

    # Inverse-distance weighted ANCC estimate (plus spread and nearest distance)
    def impute(self,xy_q,self_wid=None,k=DENSE_K,nfetch=5000):
        xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale; nf=min(nfetch,len(self.ancc))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
        sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw); an=self.ancc[ik]
        ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
        var=((an-ap[:,None])**2*w).sum(1)/safe
        return ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),np.where(vk,dk,np.inf).min(1).astype(np.float32)

# Enumerate training wells and build the two spatial imputers from them
hw_paths=sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
train_wids=[p.stem.replace('__horizontal_well','') for p in hw_paths]
FI=FormationPlaneKNN(train_wids,CFG.dataset_path / "train")
DI=DenseANCCImputer(train_wids,CFG.dataset_path / "train")

# Module-level handles to the imputers, plus GR-vs-typewell anchor offset grids
_FI=FI; _DI=DI
ANCH_OFFS=np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],np.float32)
BEAM_OFFS=np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],np.float32)
SC_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
PF_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)

# Build the full feature row-set for a single well (the heart of feature engineering)
def build_well(hw_path,tw_path,is_train):
    global _FI,_DI
    wid=Path(hw_path).stem.replace('__horizontal_well','')
    # Load horizontal + type well, bailing out on any unusable case
    try:
        hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
    except: return None
    if is_train and 'TVT' not in hw.columns: return None
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0 or len(kn)<10: return None
    if is_train and hw['TVT'].isna().all(): return None
    tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
    if len(tw_tvt)<3: return None

    # Particle-filter signals (ANCC + Z variants)
    pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr)
    if len(pf_a)==0: return None
    pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr)
    pf_use=pf_a.astype(np.float32); std_use=std_a.astype(np.float32)
    has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))

    # Last-known anchor and gap-filled GR for both prefix and eval sections
    lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)

    # 7 beams (Numba JIT +/-2)
    bpaths={}
    for (bs,mc,es,r,tag) in BEAMS:
        bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
    beam_ref=(bpaths['cons']+bpaths['sm5'])/2.

    # Multi-scale NCC -> score-weighted ensemble
    ktvt=kn['TVT_input'].to_numpy(np.float32)
    sc_res,sc_ens=multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3)
    sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]
    sc_cons=(sc8+sc15+sc25)/3.
    sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
    # Blend beam reference with NCC ensemble, trusting NCC more as prefix grows
    hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_ens  # use ensemble not single

    # Affine GR calibration and prefix-quality / slope diagnostics
    tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
    a_cal,b_cal=affine_cal(kgr,tw_at_k)
    kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
    pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
    slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
    slp_z=robust_slope(kz,ktvt)

    # Spatial formation imputation at eval and known XY positions
    swid=wid if is_train else None
    xy_ev=ev[['X','Y']].to_numpy(np.float64); xy_kn=kn[['X','Y']].to_numpy(np.float64)
    form_ev,knn_d=_FI.impute(xy_ev,self_wid=swid)
    form_kn,_   =_FI.impute(xy_kn,self_wid=swid)
    z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)

    # Per-formation: segment b_well (early/mid/late/wls) + TVT + known-zone RMSE
    tvt_fs={}; form_rmse={}; form_list=[]
    for fi2,fn in enumerate(FORMATIONS):
        b_full,b_early,b_mid,b_late,b_wls=seg_b_well(ktvt,z_kn,form_kn[:,fi2])
        tvt_f  =(-z_ev+form_ev[:,fi2]+b_full ).astype(np.float32)
        tvt_fw =(-z_ev+form_ev[:,fi2]+b_wls  ).astype(np.float32)
        tvt_f50=(-z_ev+form_ev[:,fi2]+b_late ).astype(np.float32)
        tvt_fs[f'tvtF_{fn}']=tvt_f; tvt_fs[f'tvtFw_{fn}']=tvt_fw
        tvt_fs[f'tvtF50_{fn}']=tvt_f50
        tvt_fs[f'bw_{fn}']=np.float32(b_full); tvt_fs[f'bww_{fn}']=np.float32(b_wls)
        tvt_fs[f'bw50_{fn}']=np.float32(b_late)
        tvt_fs[f'bw_early_{fn}']=np.float32(b_early)   # NEW: early segment
        tvt_fs[f'bw_mid_{fn}']=np.float32(b_mid)       # NEW: mid segment
        form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2)))
        form_list.append(tvt_f)

    # Cross-formation agreement statistics
    fs=np.stack(form_list,1)
    form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)
    form_std_d =fs.std(1).astype(np.float32)
    form_rng_d =(fs.max(1)-fs.min(1)).astype(np.float32)

    # Dense-ANCC imputation and its derived TVT estimates / residual diagnostics
    d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
    d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
    b_vd=ktvt+z_kn-d_kn
    _,b_de,b_dm,b_dl,b_dw=seg_b_well(ktvt,z_kn,d_kn)
    b_d=float(np.median(b_vd))
    tvt_dense  =(-z_ev+d_ancc+b_d  ).astype(np.float32)
    tvt_densew =(-z_ev+d_ancc+b_dw ).astype(np.float32)
    tvt_dense50=(-z_ev+d_ancc+b_dl ).astype(np.float32)
    res_kn=ktvt+z_kn-d_kn
    d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))

    # Cross-signal agreement (std/mean) across all the candidate TVT signals
    all_sigs=[pf_use]+[p for p in bpaths.values()]+[sc8,sc15,sc25,sc_ens,tvt_fs['tvtF_ANCC'],tvt_dense]
    sig_mat=np.stack(all_sigs,1)
    sig_std=sig_mat.std(1).astype(np.float32)
    sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)

    # Rolling GR statistics, lags/leads, derivatives, envelope and energy
    gr_s=pd.Series(gr_full.values); rolls={}
    for w in [5,21,51,101]:
        r=gr_s.rolling(w,center=True,min_periods=1)
        rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
        rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1,5,15,30]:
        rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)
                   ).iloc[ev.index].values.astype(np.float32)

    # Slope-extrapolated TVT baselines from the last-known point
    hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
    slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32)
    slp_b_50 =(last_tvt+slp_50 *md_since).astype(np.float32)

    # Geometric per-step gradients (dZ/dMD, dX/dMD, dY/dMD)
    mdd=hw['MD'].diff().replace(0,np.nan)
    dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)

    # Positional fraction along the eval section and a constant-broadcast helper
    nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
    def sc(v): return np.full(nh,np.float32(v),np.float32)

    # Assemble every engineered feature into one dict (later -> DataFrame)
    feats={
        'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
        'last_known_tvt':sc(last_tvt),
        'pf_ancc':pf_use,'pf_ancc_std':std_use,
        'pf_ancc_delta':(pf_use-last_tvt).astype(np.float32),
        'pf_z':(pf_z.astype(np.float32) if has_z else sc(last_tvt)),
        'pf_z_delta':((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
        'pf_vs_z':((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
        **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
        'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
        'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
        'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
        'sc8_d':(sc8-np.float32(last_tvt)).astype(np.float32),'sc8_sc':sc8s,
        'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_sc':sc15s,
        'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_sc':sc25s,
        'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),
        'sc_ens_d':(sc_ens-np.float32(last_tvt)).astype(np.float32),  # score-weighted ensemble
        'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
        'sig_std':sig_std,'sig_mean_d':sig_mean,
        **tvt_fs,
        **{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,
        'spatial_ancc_d':(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),
        'spatial_knn_dist':knn_d,
        'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
        'tvt_dense_d' :(tvt_dense -last_tvt).astype(np.float32),
        'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),
        'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),
        'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),'dense_nb_std':sc(d_nb_std),
        'pf_vs_spatial':(pf_use-tvt_fs['tvtF_ANCC']).astype(np.float32),
        'pf_vs_dense':(pf_use-tvt_dense).astype(np.float32),
        'spatial_vs_dense':(tvt_fs['tvtF_ANCC']-tvt_dense).astype(np.float32),
        'beam_vs_spatial':(bpaths['cons']-tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam':(sc_ens-bpaths['cons']).astype(np.float32),
        'cal_a':sc(a_cal),'cal_b':sc(b_cal),
        'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
        'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
        'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),
        'slp_b_d_50': (slp_b_50 -last_tvt).astype(np.float32),
        'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),
        'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),
        'z':z_ev,
        'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),
        'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),
        'dz':(z_ev-float(lk['Z'])).astype(np.float32),
        'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),
        'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
        'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,'gr_env':gr_env,'gr_nrg':gr_nrg,
        'gr_vs_tw_anc':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),
        'gr_vs_slp_all':hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
        **{f'tda{int(o)}' :hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
        **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f'tdsc{int(o)}':hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
        **{f'tdpf{int(o)}':hgr-np.interp(pf_use+o,tw_tvt,tw_gr).astype(np.float32) for o in PF_OFFS},
        'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),
    }
    # Merge in the rolling-GR features
    for k,v in rolls.items(): feats[k]=v
    result=pd.DataFrame(feats)
    # For training rows, attach the regression target (TVT minus last-known)
    if is_train:
        if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
        result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
    return result

# Build the full dataset by running build_well over all wells in parallel
def build_dataset(paths,is_train,label):
    args=[(str(p),str(p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv'),is_train)
          for p in paths
          if (p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv').exists()]
    t0=time.time()
    res=Parallel(n_jobs=NCPU,prefer='threads',verbose=3)(
        delayed(build_well)(hp,tp,it) for hp,tp,it in args)
    parts=[r for r in res if r is not None]
    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

In [ ]:
# Use cached training features if available, otherwise build them from scratch
if (CFG.artifacts_path / "data" / "train.csv").exists():
    # Load the precomputed training feature table
    train_df = pd.read_csv(CFG.artifacts_path / "data" / "train.csv", low_memory=False)
else:
    # Find every training horizontal-well file
    train_paths = sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
    # Build the training features in parallel
    train_df = build_dataset(train_paths, is_train=True, label="train")

# Find every test horizontal-well file
test_paths = sorted((CFG.dataset_path / "test").glob('*__horizontal_well.csv'))
# Build the test features
test_df = build_dataset(test_paths, is_train=False, label="test")

# Feature columns are everything except the id/group/target columns
features = [c for c in train_df.columns if c not in {'well','id','target'}]

# Training design matrix
X = train_df[features]
# Training target (TVT delta from last-known)
y = train_df['target']
# Group labels (well id) for grouped cross-validation
g = train_df['well']

# Test design matrix
X_test = test_df[features]

In [ ]:
# LightGBM hyper-parameter sets; the ensemble trains one model per dict
lgb_params = [
    # Model 1: many leaves, moderate regularisation, fast-ish learning rate
    dict(
        boosting_type="gbdt", 
        num_leaves=255, 
        min_child_samples=15,
        subsample=0.8, 
        subsample_freq=1, 
        colsample_bytree=0.8,
        reg_lambda=3.0, 
        reg_alpha=0.05, 
        objective="regression",
        verbose=-1, 
        n_jobs=-1, 
        device_type="gpu", 
        gpu_use_dp=False, 
        max_bin=255,
        learning_rate=0.030, 
        n_estimators=5000, 
        seed=123
    ),
    # Model 2: heavily regularised, slow learning rate, seed 0
    dict(
        n_jobs=-1, 
        verbose=-1, 
        reg_alpha=10.788188919840913, 
        subsample=0.47437582748953966, 
        num_leaves=64, 
        reg_lambda=95.75401894533888, 
        n_estimators=10000,
        random_state=0,
        boosting_type='gbdt', 
        learning_rate=0.00934485794382918,
        colsample_bytree=0.39283351290380497,
        min_child_weight=0.24081152127177283, 
        min_child_samples=40,
        device='gpu',
    ),
    # Model 3: same as model 2 but a different seed for ensemble diversity
    dict(
        n_jobs=-1, 
        verbose=-1, 
        reg_alpha=10.788188919840913, 
        subsample=0.47437582748953966, 
        num_leaves=64, 
        reg_lambda=95.75401894533888, 
        n_estimators=10000,
        random_state=29,
        boosting_type='gbdt', 
        learning_rate=0.00934485794382918,
        colsample_bytree=0.39283351290380497,
        min_child_weight=0.24081152127177283, 
        min_child_samples=40,
        device='gpu',
    ),
]

# CatBoost hyper-parameter sets (one model per dict)
cb_params = [
    # Model 1: depth-7 trees, slower learning rate, seed 7
    dict(
        iterations=8000, 
        depth=7, 
        l2_leaf_reg=2.0,
        min_data_in_leaf=15, 
        border_count=254,
        loss_function="RMSE", 
        task_type="GPU", 
        devices="0",
        od_type="Iter", 
        od_wait=300, 
        verbose=0,
        learning_rate=0.020, 
        random_seed=7
    ),
    # Model 2: same depth, faster learning rate, seed 123
    dict(
        iterations=8000, 
        depth=7, 
        l2_leaf_reg=2.0,
        min_data_in_leaf=15, 
        border_count=254,
        loss_function="RMSE", 
        task_type="GPU", 
        devices="0",
        od_type="Iter", 
        od_wait=300, 
        verbose=0,
        learning_rate=0.030, 
        random_seed=123
    ),
]

# Ridge meta-learner settings (non-negative weights, fitted intercept)
ridge_params = {
    "random_state": 42,
    "alpha": 1.6602834637650032,
    "tol": 0.0005030247295617308,
    "positive": True,
    "fit_intercept": True
}

# Post-processing blend settings: shrinkage alpha, decay tau, particle-filter weight
pp_params = {
    'alpha': 1.0,
    'tau': 85,
    'w_pf': 0.09
}

In [ ]:
# Out-of-fold predictions per base model (for stacking)
oof_preds = {}
# Test predictions per base model
test_preds = {}

# Overall CV score per base model
overall_scores = {}
# Per-fold scores per base model
fold_scores = {}

In [ ]:
# Train (or load) one LightGBM model per parameter set
for i, params in enumerate(lgb_params):
    # Where this model's artifacts live on disk
    save_path = f"models/lightgbm-{i+1}"

    # Reuse a saved model if present, otherwise train a fresh one
    if (CFG.artifacts_path / save_path).exists():
        # Announce that we are loading from cache
        print(f"Loading lightgbm-{i+1} from disk...")

        # Locate the pickled trainer file
        trainer_paths = (CFG.artifacts_path / save_path).glob('*.pkl')
        # Load the trainer (with its OOF preds and scores)
        trainer = joblib.load(list(trainer_paths)[0])

        # Report the loaded model's overall score
        print(f"Loaded lightgbm-{i+1} with overall RMSE: {trainer.overall_score:.4f}\n")
    else:
        # Configure a grouped-CV trainer around this LightGBM estimator
        trainer = Trainer(
            estimator=LGBMRegressor(**params),
            task="regression",
            metric=CFG.metric,
            cv=CFG.cv,
            cv_args={"groups": g},
            use_early_stopping=True,
            verbose=True,
            save=True,
            save_path=save_path
        )

        # Fit with RMSE eval, periodic logging, and early stopping
        trainer.fit(
            X, 
            y,
            fit_args={
                "eval_metric": "rmse",
                "callbacks": [
                    log_evaluation(period=250), 
                    early_stopping(stopping_rounds=250)
                ]
            }
        )
        # Spacer in the training log
        print("\n\n")

    # Store this model's out-of-fold predictions
    oof_preds[f"lightgbm-{i+1}"] = trainer.oof_preds
    # Store its test predictions
    test_preds[f"lightgbm-{i+1}"] = trainer.predict(X_test)
    # Store its overall score
    overall_scores[f"lightgbm-{i+1}"] = trainer.overall_score
    # Store its per-fold scores
    fold_scores[f"lightgbm-{i+1}"] = trainer.fold_scores

In [ ]:
# Train (or load) one CatBoost model per parameter set
for i, params in enumerate(cb_params):
    # Where this model's artifacts live on disk
    save_path = f"models/catboost-{i+1}"
    # Reuse a saved model if present, otherwise train a fresh one
    if (CFG.artifacts_path / save_path).exists():
        # Announce that we are loading from cache
        print(f"Loading catboost-{i+1} from disk...")

        # Locate the pickled trainer file
        trainer_paths = (CFG.artifacts_path / save_path).glob('*.pkl')
        # Load the trainer (with its OOF preds and scores)
        trainer = joblib.load(list(trainer_paths)[0])

        # Report the loaded model's overall score
        print(f"Loaded catboost-{i+1} with overall RMSE: {trainer.overall_score:.4f}\n")
    else:
        # Configure a grouped-CV trainer around this CatBoost estimator
        trainer = Trainer(
            estimator=CatBoostRegressor(**params),
            task="regression",
            metric=CFG.metric,
            cv=CFG.cv,
            cv_args={"groups": g},
            use_early_stopping=True,
            verbose=True,
            save=True,
            save_path=save_path
        )

        # Fit with periodic logging, early stopping, and best-model selection
        trainer.fit(
            X, 
            y,
            fit_args={
                "verbose": 250,
                "early_stopping_rounds": 250,
                "use_best_model": True
            }
        )
        # Spacer in the training log
        print("\n\n")

    # Store this model's out-of-fold predictions
    oof_preds[f"catboost-{i+1}"] = trainer.oof_preds
    # Store its test predictions
    test_preds[f"catboost-{i+1}"] = trainer.predict(X_test)
    # Store its overall score
    overall_scores[f"catboost-{i+1}"] = trainer.overall_score
    # Store its per-fold scores
    fold_scores[f"catboost-{i+1}"] = trainer.fold_scores

In [ ]:
# Turn the per-model OOF prediction dict into a DataFrame (one column per model)
oof_preds = pd.DataFrame(oof_preds)
# Same for the test predictions
test_preds = pd.DataFrame(test_preds)

In [ ]:
# Ridge meta-learner that stacks the base models' OOF predictions
ridge_trainer = Trainer(
    Ridge(**ridge_params),
    task="regression",
    metric=CFG.metric,
    cv=CFG.cv,
    cv_args={"groups": g},
    verbose=True
)

# Fit the stacker on base-model OOF predictions against the true target
ridge_trainer.fit(oof_preds, y)

# Stacked out-of-fold predictions
ridge_oof_preds = ridge_trainer.oof_preds
# Stacked test predictions
ridge_test_preds = ridge_trainer.predict(test_preds)

# Record the stacker's overall and per-fold scores
overall_scores["ridge"] = ridge_trainer.overall_score
fold_scores["ridge"] = ridge_trainer.fold_scores

In [ ]:
# Post-process a model delta: blend with the PF signal, then ramp + scale it
def apply_pp(df, md, pd_, alpha, tau, w_pf):
    # Blend the model delta with the particle-filter delta
    d = md * (1-w_pf) + pd_ * w_pf
    # Optionally ramp the delta up over MD-since-last-known with time-constant tau
    if tau: 
        d *= (1.-np.exp(-np.maximum(df['md_since'].values,0.) / tau))
    # Apply the global shrinkage factor
    return d * alpha

# Savitzky-Golay smooth a column independently within each well
def sg_smooth(df, col, sg_w=17, sg_p=3):
    # Work on a copy so the caller's frame is untouched
    df = df.copy()
    # Smooth each well's series separately (order preserved)
    for _, g in df.groupby('well', sort=False):
        # Values to smooth
        v = g[col].values
        # Series length
        n = len(v)
        # Window cannot exceed the series length
        wl = min(sg_w, n)
        # Savitzky-Golay needs an odd window
        if wl % 2 == 0: 
            wl -= 1
        # Only smooth when the window is large enough for the polynomial order
        if wl >= sg_p + 2: 
            v = savgol_filter(v, wl, sg_p)
        # Write the smoothed values back in place
        df.loc[g.index,col] = v
    # Return the smoothed frame
    return df

In [ ]:
# Last-known TVT baseline per row
base = train_df['last_known_tvt'].values
# Reconstruct absolute true TVT from the delta target plus the baseline
ytrue = y.values + base

# Particle-filter delta relative to the baseline (OOF analogue)
pf_oof = (train_df['pf_ancc'].values - base)

# Post-processed stacked delta on the training OOF predictions
d = apply_pp(train_df, ridge_oof_preds, pf_oof, **pp_params)
# Score the post-processed prediction against the true TVT
ridge_score = root_mean_squared_error(ytrue, base + d)

# Record the post-processed score overall and (flat) per fold
overall_scores["ridge (pp)"] = ridge_score
fold_scores["ridge (pp)"] = [ridge_score] * CFG.n_splits

In [ ]:
# Work on a copy of the test features
test_df2 = test_df.copy()
# Particle-filter delta relative to the last-known baseline on test
pf_test = test_df2['pf_ancc'].values - test_df2['last_known_tvt'].values

# Absolute test prediction = baseline + post-processed stacked delta
test_df2['pred'] = test_df2['last_known_tvt'].values + apply_pp(
    test_df2, 
    ridge_test_preds,
    pf_test, 
    **pp_params
)
# Smooth the prediction within each well
test_df2 = sg_smooth(test_df2, 'pred')

In [ ]:
# Read the submission template to fix row order and ids
sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
# Attach predictions to each template id (renaming pred -> tvt)
sub_1 = (sample_sub[['id']].merge(
    test_df2[['id', 'pred']].rename(columns={'pred':'tvt'}),
    on='id', 
    how='left'
))

# Fill any missing predictions with a global fallback TVT estimate
sub_1['tvt']=sub_1['tvt'].fillna(float(train_df['last_known_tvt'].mean()+train_df['target'].mean()))
# Display the assembled submission
sub_1

In [ ]:
# Load the submission template and derive well id + row index from each id
sample = pd.read_csv(CFG.dataset_path / 'sample_submission.csv')
sample['well']    = sample['id'].str[:8]
sample['row_idx'] = sample['id'].str[9:].astype(int)

# List training wells (used to detect "visible" wells with known geology)
train_hw_files = sorted(glob.glob(str(CFG.dataset_path / 'train' / '*__horizontal_well.csv')))
train_wells = [os.path.basename(f).split('__')[0] for f in train_hw_files]

# List test wells to process
test_hw_files = sorted(glob.glob(str(CFG.dataset_path / 'test' / '*__horizontal_well.csv')))
test_wells = [os.path.basename(f).split('__')[0] for f in test_hw_files]

# Accumulate one prediction row per submission id
rows = []
# Process every test well in turn
for i, wid in enumerate(test_wells):
    # Progress log
    print(f'\nProcessing {i + 1}/{len(test_wells)}: {wid}...')
    # Load this well's horizontal and type-well tables
    hw_te, tw_te = load_well(wid, 'test')

    # Physical-model TVT (only available for visible wells)
    tvt_phys = None
    # Training horizontal well (if visible)
    hw_tr    = None
    # Training type well (if visible)
    tw_tr    = None

    # Physical model for visible wells
    if wid in train_wells:
        # Try to build the contact-based physical TVT from the training data
        try:
            # Load the training tables for this well
            hw_tr, tw_tr = load_well(wid, 'train')
            # Copy the known TVT inputs over to the test frame
            hw_te['TVT_input'] = hw_tr['TVT_input'].values
            # Derive TVT directly from formation contacts
            tvt_phys = tvt_from_contacts(hw_tr, tw_tr)
            # Note success
            print(f'  Physical model OK')
        except Exception as e:
            # Fall back if the physical model fails
            print(f'  Physical model failed: {e}')
            tvt_phys = None

    # Classify the well and pick its blending variant
    selector_code, selector_variant, selector_n_eval, selector_z_span = selector_well_code(hw_te)

    # 128-seed likelihood-weighted PF ensemble
    try:
        # Use the training type well when available, else the test one
        tw_ref = tw_tr if tw_tr is not None else tw_te
        # Run the multi-scale likelihood-weighted particle-filter ensemble
        pf_by_scale = run_pf_lik_ensemble_scales(hw_te, tw_ref, n_particles=500, n_seeds=128)
        # Default single-scale PF prediction
        tvt_pf = pf_by_scale['pf_scale_8']
        # Note success
        print(f'  PF 128-seed lik-ensemble OK scales={SELECTOR_SCALES}')
    except Exception as e:
        # On failure, fall back to holding the last-known TVT forward
        print(f'  PF failed: {e}')
        last_known = hw_te['TVT_input'].dropna()
        last_val   = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0
        tvt_pf = hw_te['TVT_input'].fillna(last_val).values.astype(float)
        pf_by_scale = {f'pf_scale_{scale:g}': tvt_pf.copy() for scale in SELECTOR_SCALES}

    # Beam search ensemble
    try:
        # Reference type well (training preferred)
        tw_ref = tw_tr if tw_tr is not None else tw_te
        # Run the 14-config beam ensemble
        tvt_beam = run_beam_ensemble(hw_te, tw_ref)
        # Note success
        print(f'  Beam 14-config ensemble OK')
    except Exception as e:
        # Fall back to the PF prediction if beam search fails
        print(f'  Beam failed: {e}')
        tvt_beam = tvt_pf.copy()

    # Selector blending
    # Last-known TVT for the "hold" term
    last_known = hw_te['TVT_input'].dropna()
    last_known_tvt = float(last_known.iloc[-1]) if len(last_known) > 0 else float(np.nanmean(tvt_pf))
    # Blend PF/beam/hold according to the chosen variant
    tvt_selector = apply_selector_variant(selector_variant, pf_by_scale, tvt_beam, last_known_tvt)
    # Log the selector decision and its features
    print(
        f'  Selector code={selector_code} variant={selector_variant} '
        f'n_eval={selector_n_eval:.0f} z_span={selector_z_span:.3f}'
    )

    # Emit one row per submission id for this well
    ws = sample[sample['well'] == wid]
    for _, row in ws.iterrows():
        # Row index within the well
        ridx = int(row['row_idx'])
        # Prefer the physical model where available, else the selector blend
        if tvt_phys is not None:
            tvt_val = float(tvt_phys.iloc[ridx])
        else:
            tvt_val = float(tvt_selector[ridx])
        # Record the prediction
        rows.append({'id': row['id'], 'tvt': tvt_val})
    # Note how many rows were added for this well
    print(f'  Added {len(ws)} rows')

In [ ]:
# Collect the per-well physical/selector predictions into a DataFrame
sub_2 = pd.DataFrame(rows)

In [ ]:
# Blend the two submissions: 30% stacked-model, 70% physical/selector
sub = (
    sub_1.merge(sub_2, on='id', suffixes=('_1', '_2'))
       .assign(tvt=lambda x: 0.3 * x['tvt_1'] + 0.7 * x['tvt_2'])
       [['id', 'tvt']]
)
# Write the blended submission to disk
sub.to_csv("submission.csv", index=False)
# Display it
sub

In [ ]:
# === robust low-order PROJECTION post-processing (patched degree=4, blend=0.75) (CV-validated: raw PF -0.54, deployed components -0.33) ===
# Runs AFTER the 0.3*ridge+0.7*selector blend writes submission.csv; OVERWRITES it with the projected
# version. Per-well robust deg-5 fit of dU = tvt + Z - anchor vs normalized MD -> denoise jitter +
# down-weight wrong-branch outliers. Deterministic; defensive per-well fallback to raw.
# Local aliases for numpy/pandas to avoid clobbering notebook globals
import numpy as _np, pandas as _pd
# Iteratively-reweighted robust polynomial fit (Cauchy weights)
def _robfit(s, y, deg=5):
    # Too few points to fit -> return the input unchanged
    if len(s) < deg + 2:
        return y.copy()
    # Initial least-squares polynomial
    c = _np.polyfit(s, y, deg)
    # Four reweighting passes to suppress outliers
    for _ in range(4):
        # Residuals of the current fit
        r = y - _np.polyval(c, s)
        # Robust scale estimate (MAD-based)
        sc = _np.median(_np.abs(r)) * 1.4826 + 1e-6
        # Refit with Cauchy-style weights that down-weight large residuals
        c = _np.polyfit(s, y, deg, w=1.0 / (1.0 + (r / (2.0 * sc)) ** 2))
    # Evaluate the final polynomial on the inputs
    return _np.polyval(c, s)
# Apply the projection well-by-well, falling back to raw on any failure
try:
    _base = _pd.read_csv("submission.csv")   # the just-written blended submission
    # Sanity-check the expected columns are present
    assert set(['id','tvt']).issubset(_base.columns)
    # Recover well id and row index from each submission id
    _base['well'] = _base['id'].str[:8]
    _base['row_idx'] = _base['id'].str[9:].astype(int)
    # Start from the raw values; overwrite per well where projection succeeds
    _out = dict(zip(_base['id'].values, _base['tvt'].astype(float).values))
    # Count of wells successfully projected
    _n_ok = 0
    # Process each well group
    for _wid, _g in _base.groupby('well'):
        try:
            # Load the test horizontal well for geometry
            _hw = _pd.read_csv(CFG.dataset_path / 'test' / (_wid + '__horizontal_well.csv'))
            # Known (conditioning) rows
            _kn = _hw[_hw['TVT_input'].notna()]
            # Need a few known rows to anchor the fit
            if len(_kn) < 5:
                continue
            # Last known row and its stratigraphic anchor (TVT + Z)
            _last = _kn.iloc[-1]
            _anchor = float(_last['TVT_input']) + float(_last['Z'])
            # MD at the prediction start and at the well end
            _ps = float(_last['MD']); _end = float(_hw['MD'].iloc[-1])
            # Sort this well's rows by index
            _gi = _g.sort_values('row_idx')
            _ri = _gi['row_idx'].values
            # Depth and MD for the predicted rows
            _Z = _hw['Z'].values[_ri].astype(float)
            _md = _hw['MD'].values[_ri].astype(float)
            # Normalised MD coordinate in [0, 1] across the eval section
            _s = (_md - _ps) / max(_end - _ps, 1e-6)
            # Current predicted TVT values
            _tvt = _gi['tvt'].values.astype(float)
            # Robustly fit the stratigraphic offset vs normalised MD
            _fit = _robfit(_s, (_tvt + _Z) - _anchor, 4)
            # Convert the fitted offset back to absolute TVT
            _tvt_fit_full = (_anchor + _fit) - _Z
            # Blend 25% raw with 75% projected
            _tvt_fit = 0.25 * _tvt + 0.75 * _tvt_fit_full
            # Skip if the fit produced anything non-finite
            if not _np.all(_np.isfinite(_tvt_fit)):
                continue
            # Write the projected values for this well
            for _rid, _val in zip(_gi['id'].values, _tvt_fit):
                _out[_rid] = float(_val)
            # Mark this well as projected
            _n_ok += 1
        except Exception as _e:
            # Per-well fallback: keep the raw value, log the reason
            print('proj fallback', _wid, _e)
    # Report how many wells were projected
    print('projection applied to', _n_ok, 'wells')
    # Rebuild the submission in original id order
    _final = _base[['id']].copy()
    _final['tvt'] = _final['id'].map(_out).astype(float)
    # Overwrite submission.csv with the projected version
    _final[['id','tvt']].to_csv("submission.csv", index=False)
    # Confirm the write
    print('wrote projected submission.csv', _final.shape)
except Exception as _e:
    # If anything failed globally, keep the blended submission untouched
    print('PROJECTION SKIPPED (kept blended submission):', _e)

In [ ]:
# Per-fold scores as a DataFrame for plotting
fold_scores_df = pd.DataFrame(fold_scores)
# Overall scores, transposed and sorted best-first
overall_scores_df = pd.DataFrame({k: [v] for k, v in overall_scores.items()}).transpose().sort_values(by=0, ascending=True)
# Model order (best to worst) for consistent axis ordering
order = overall_scores_df.index.tolist()

# Compute padded x-limits for the overall-score bar chart
min_score = overall_scores_df.values.flatten().min()
max_score = overall_scores_df.values.flatten().max()
padding = (max_score - min_score) * 0.5
lower_limit = min_score - padding
upper_limit = max_score + padding

# Two side-by-side panels: fold boxplots and overall bars
fig, axs = plt.subplots(1, 2, figsize=(15, fold_scores_df.shape[1] * 0.5))

# Left panel: distribution of fold RMSE per model
boxplot = sns.boxplot(data=fold_scores_df, order=order, ax=axs[0], orient="h", color="grey")
axs[0].set_title(f"Fold RMSE")
axs[0].set_xlabel("")
axs[0].set_ylabel("")

# Right panel: overall RMSE per model
barplot = sns.barplot(x=overall_scores_df.values.flatten(), y=overall_scores_df.index, ax=axs[1], color="grey")
axs[1].set_title(f"Overall RMSE")
axs[1].set_xlabel("")
axs[1].set_xlim(left=lower_limit, right=upper_limit)
axs[1].set_ylabel("")

# Highlight ridge models in cyan and annotate each bar with its score
for i, (score, model) in enumerate(zip(overall_scores_df.values.flatten(), overall_scores_df.index)):
    color = "cyan" if "ridge" in model.lower() else "grey"
    barplot.patches[i].set_facecolor(color)
    boxplot.patches[i].set_facecolor(color)
    barplot.text(score, i, round(score, 3), va="center")

# Finalise and render the figure
plt.tight_layout()
plt.show()

In [ ]:
# Local Path/pandas aliases for this self-contained blending snippet
from pathlib import Path as _BlendPath
import pandas as _blend_pd
# Locate the current submission (Kaggle working dir or local)
_sp45_path = _BlendPath('/kaggle/working/submission.csv') if _BlendPath('/kaggle/working').exists() else _BlendPath('submission.csv')
# Read it back
_sp45_df = _blend_pd.read_csv(_sp45_path)
# Save a named snapshot of the projection submission
_sp45_df.to_csv((_BlendPath('/kaggle/working') if _BlendPath('/kaggle/working').exists() else _BlendPath('.')) / 'sp45_projection_submission.csv', index=False)
# Confirm the snapshot was written
print('saved sp45_projection_submission.csv', _sp45_df.shape, flush=True)

In [ ]:
# Standard-library and scientific imports for the self-contained inference module
import os, sys, glob, time, warnings, multiprocessing
from pathlib import Path
import numpy as np
import pandas as pd
from numba import njit
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from joblib import Parallel, delayed
# Silence warnings and default to no figures unless overridden
warnings.filterwarnings("ignore")
os.environ.setdefault("SHOW_FIGS", "0")

# ---- environment / paths (Kaggle or local) -------------------------------------
# Locate the competition data directory across known mount points
def _find_data():
    # Check the standard Kaggle competition paths first
    for c in ["/kaggle/input/competitions/rogii-wellbore-geology-prediction",
              "/kaggle/input/rogii-wellbore-geology-prediction"]:
        if Path(c).exists() and (Path(c)/"train").exists():
            return Path(c)
    # fallback: find any mounted folder that contains a train/ directory
    for p in glob.glob("/kaggle/input/**/train", recursive=True):
        return Path(p).parent
    # Final fallback to an env var or the current directory
    return Path(os.environ.get("ROGII_DATA", "."))   # local override for development

# Self-contained configuration for the inference module
class CFG:
    # Resolved data directory
    DATA = _find_data()
    # Output directory (Kaggle working dir or local)
    OUT  = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
    # Random seed and CV fold count
    seed = 42
    n_splits = 5
    # Worker count, capped at 8
    n_jobs = min(8, multiprocessing.cpu_count())
    # lik-PF
    # Particle-filter ensemble settings
    PF_SEEDS = 128
    PF_PARTICLES = 500
    PF_SCALES = (3., 5., 8., 12.)
    # FAST dev (local smoke test): limit train wells & trees
    FAST = bool(int(os.environ.get("FAST", "0")))
    N_TRAIN_WELLS = int(os.environ.get("N_TRAIN_WELLS", "0"))  # 0 = all
    # GPU and figure toggles
    USE_GPU = os.environ.get("USE_GPU", "auto")
    SHOW_FIGS = os.environ.get("SHOW_FIGS", "1") == "1"   # EDA plots (on in the notebook)

# Formation columns reused by this module
FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
# Pick a representative training well for the EDA figures
def _demo_well():
    """A train well with TVT + a sizable eval zone, for the EDA plots."""
    # Scan train wells in id order
    for w in sorted(p.stem.replace("__horizontal_well", "")
                    for p in (CFG.DATA/"train").glob("*__horizontal_well.csv")):
        try:
            # Read just the TVT columns to test suitability
            d = pd.read_csv(CFG.DATA/"train"/f"{w}__horizontal_well.csv", usecols=["TVT", "TVT_input"])
        except Exception:
            continue
        # Require ground-truth TVT and a large eval zone
        if "TVT" in d and d.TVT.notna().any() and d.TVT_input.isna().sum() > 2000:
            return w
    # None found
    return None
# Report the resolved environment
print("DATA:", CFG.DATA, "| OUT:", CFG.OUT, "| cores:", CFG.n_jobs, "| FAST:", CFG.FAST)

# Load horizontal + (TVT-sorted) type well for a given id
def load_well(wid, split="train"):
    base = CFG.DATA / split
    hw = pd.read_csv(base / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(base / f"{wid}__typewell.csv").sort_values("TVT")
    return hw, tw

# Root-mean-squared error helper
def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float))**2)))

In [ ]:
# Three-panel overview: trajectory, gamma-ray, and TVT target for one well
def fig_overview(wid):
    import matplotlib.pyplot as plt
    hw, tw = load_well(wid)
    kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]; ps = kn.MD.iloc[-1]
    fig, ax = plt.subplots(3, 1, figsize=(12, 8.5), sharex=True)
    ax[0].plot(hw.MD, hw.Z, lw=1.2, color="#333"); ax[0].axvline(ps, color="crimson", ls="--", label="PS")
    ax[0].set_ylabel("Z / TVD (ft)"); ax[0].legend(loc="upper right")
    ax[0].set_title(f"Well {wid}: trajectory Â· gamma-ray Â· TVT target")
    ax[1].plot(kn.MD, kn.GR, lw=.7, color="steelblue", label="GR known")
    ax[1].plot(ev.MD, ev.GR, lw=.7, color="darkorange", label="GR eval"); ax[1].axvline(ps, color="crimson", ls="--")
    ax[1].set_ylabel("GR (API)"); ax[1].legend(loc="upper right")
    ax[2].plot(kn.MD, kn.TVT, lw=1.6, color="seagreen", label="TVT known (=input)")
    ax[2].plot(ev.MD, ev.TVT, lw=1.6, color="crimson", label="TVT to predict"); ax[2].axvline(ps, color="crimson", ls="--")
    ax[2].set_ylabel("TVT (ft)"); ax[2].set_xlabel("MD (ft)"); ax[2].invert_yaxis(); ax[2].legend(loc="upper right")
    for a in ax: a.grid(alpha=.25)
    plt.tight_layout(); plt.show()

# Show that horizontal GR at its true TVT matches the type-well GR signature
def fig_correlation(wid):
    import matplotlib.pyplot as plt
    hw, tw = load_well(wid); ev = hw[hw.TVT_input.isna()]
    fig, ax = plt.subplots(1, 2, figsize=(11, 6))
    ax[0].plot(tw.GR, tw.TVT, lw=1.0, color="black")
    ax[0].set_xlabel("GR (API)"); ax[0].set_ylabel("TVT (ft)"); ax[0].invert_yaxis()
    ax[0].set_title("Typewell signature: GR vs TVT")
    sc = ax[1].scatter(ev.GR, ev.TVT, s=4, c=ev.MD, cmap="viridis")
    ax[1].set_xlabel("GR (API)"); ax[1].set_ylabel("TVT (ft)"); ax[1].invert_yaxis()
    ax[1].set_title("Horizontal GR at its true TVT\nmatches the typewell signature")
    plt.colorbar(sc, ax=ax[1], label="MD (ft)")
    for a in ax: a.grid(alpha=.25)
    plt.tight_layout(); plt.show()

# Quantify how a few high-drift wells dominate the pooled error metric
def fig_drift_tail(n_wells=250):
    import matplotlib.pyplot as plt
    wids = sorted(p.stem.replace("__horizontal_well", "") for p in (CFG.DATA/"train").glob("*__horizontal_well.csv"))
    rng = np.random.default_rng(1); samp = sorted(rng.choice(wids, min(n_wells, len(wids)), replace=False).tolist())
    per = []
    for wid in samp:
        try: hw = pd.read_csv(CFG.DATA/"train"/f"{wid}__horizontal_well.csv", usecols=["TVT_input", "TVT"])
        except: continue
        ev = hw[hw.TVT_input.isna()]; kn = hw[hw.TVT_input.notna()]
        if len(ev) == 0 or len(kn) < 10 or hw.TVT.isna().all(): continue
        t = ev.TVT.values
        if np.isnan(t).any(): continue
        per.append(np.sqrt(np.mean((t-kn.TVT_input.iloc[-1])**2)))
    per = np.array(per); srt = np.sort(per)[::-1]; cum = np.cumsum(srt**2)/np.sum(srt**2)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].hist(per, bins=40, color="indianred", alpha=.85)
    ax[0].axvline(np.median(per), color="k", ls="--", label=f"median={np.median(per):.1f}")
    ax[0].axvline(per.mean(), color="b", ls="--", label=f"mean={per.mean():.1f}")
    ax[0].set_xlabel("per-well last-known-baseline RMSE (ft)"); ax[0].set_ylabel("wells"); ax[0].legend()
    ax[0].set_title("Per-well error is heavily right-skewed")
    ax[1].plot(np.arange(1, len(srt)+1)/len(srt)*100, cum*100, color="purple"); ax[1].axhline(80, color="gray", ls=":")
    ax[1].set_xlabel("% of wells (worst first)"); ax[1].set_ylabel("% of pooled squared error")
    ax[1].set_title("A few drift wells dominate the metric")
    for a in ax: a.grid(alpha=.25)
    plt.tight_layout(); plt.show()

In [ ]:
# Choose a fixed demo well if present, else auto-select one
DEMO = "00bbac68" if (CFG.DATA/"train"/"00bbac68__horizontal_well.csv").exists() else _demo_well()
# Render the EDA figures only when figures are enabled
if CFG.SHOW_FIGS:
    print("demo well:", DEMO)
    if DEMO:
        fig_overview(DEMO)
        fig_correlation(DEMO)
    fig_drift_tail()

In [ ]:
# ---- single particle filters (ANCC-anchored & Z-velocity-coupled), numba ---------
PF_N = 600; ANCC_N = 600
PF_MOM = 0.993; PF_VN = 0.005; PF_PN = 0.01
PF_GR_SIG_MIN = 10.; PF_GR_SIG_MAX = 60.; PF_GR_SIG_DEF = 30.
PF_GR_WIN = 5; PF_GR_WT = 0.3; PF_RESAMP = 0.5; PF_ROUGH_P = 0.2; PF_ROUGH_V = 0.003
ANCC_ALPHA = 0.998; ANCC_RN = 0.002; ANCC_PN = 0.005; ANCC_IS = 0.3; ANCC_RP = 0.1; ANCC_RR = 0.001

BEAMS = [(10,20.,144.,2,"cons"),(10,8.,64.,2,"loose"),(8,35.,220.,1,"vcons"),
         (10,14.,90.,5,"sm5"),(20,4.,36.,3,"vloose"),(12,12.,100.,3,"mid"),(15,25.,180.,2,"stiff")]

# JIT: linear interpolation on a uniform grid (clamped at the ends)
@njit(cache=True)
# JIT: linear interpolation on a uniform grid (clamped at the ends)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i]*(1.-t) + grid[i+1]*t

# JIT: systematic resampling of (position, aux) particles with roughening
@njit(cache=True)
# JIT: systematic resampling of (position, aux) particles with roughening
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N+1)
    for j in range(N): cum[j+1] = cum[j]+w[j]
    u0 = np.random.uniform(0., 1./N); np2 = np.empty(N); na = np.empty(N); ci = 0
    for j in range(N):
        u = u0+j/N
        while ci < N-1 and cum[ci+1] < u: ci += 1
        np2[j] = pos[ci]+rp*np.random.randn(); na[j] = aux[ci]+rv*np.random.randn()
    return np2, na

# JIT: beam search over +/-2 index moves vs the type-well GR; backtracks the best path
@njit(cache=True)
# JIT: beam search over +/-2 index moves vs the type-well GR; backtracks the best path
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS*6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx+d
                if ni < 0 or ni >= nt: continue
                tot = cost+(gv-tw_gr[ni])**2/es+mc*(d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i+1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]; cC[i], cC[mi] = cC[mi], cC[i]; cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n-1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path

# JIT: ANCC-anchored particle filter -> posterior-mean TVT and per-row std
@njit(cache=True)
# JIT: ANCC-anchored particle filter -> posterior-mean TVT and per-row std
def _pf_ancc(md_v, z_v, gr_v, gg, vmin, step, gs, ls, ir, N, ALPHA, RN, PN, IS, RP, RR, RESAMP):
    pos = np.empty(N); rate = np.empty(N); w = np.ones(N)/N
    for j in range(N):
        pos[j] = ls+IS*np.random.randn(); rate[j] = ir+0.01*np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0]-1.
    for i in range(len(md_v)):
        dm = md_v[i]-pm; dm = max(dm, 1.)
        for j in range(N):
            rate[j] = ALPHA*rate[j]+RN*np.random.randn(); pos[j] += rate[j]*dm+PN*np.random.randn()
            tvt_j = pos[j]-z_v[i]; tvt_j = max(tvt_j, vmin-50.); tvt_j = min(tvt_j, vmin+len(gg)*step+50.)
            pos[j] = tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                eg = _interp1(gg, pos[j]-z_v[i], vmin, step); d = (gr_v[i]-eg)/gs
                lk = max(np.exp(-0.5*d*d) if d*d < 600. else 0., 1e-300); w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1./N
        ne = 0.
        for j in range(N): ne += w[j]*w[j]
        if 1./ne < RESAMP*N:
            pos, rate = _resamp(pos, rate, w, N, RP, RR)
            for j in range(N): w[j] = 1./N
        tv = 0.
        for j in range(N): tv += w[j]*(pos[j]-z_v[i])
        pts[i] = tv; va = 0.
        for j in range(N): va += w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i] = va**0.5; pm = md_v[i]
    return pts, std_

# JIT: Z-velocity-coupled particle filter -> posterior-mean TVT and per-row std
@njit(cache=True)
# JIT: Z-velocity-coupled particle filter -> posterior-mean TVT and per-row std
def _pf_z(md_v, z_v, gr_v, gr_sm_v, gg_p, gg_s, vmin, step, gs, ip, iv, beta, icpt, zsig, N,
         MOM, VN, PN, GR_WT, RP, RV, RESAMP):
    pos = np.empty(N); vel = np.empty(N); w = np.ones(N)/N
    for j in range(N):
        pos[j] = ip+0.5*np.random.randn(); vel[j] = iv+0.02*np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0]-1.; pz = z_v[0]-1.
    for i in range(len(md_v)):
        dm = md_v[i]-pm; dm = max(dm, 1.); dzd = (z_v[i]-pz)/dm; ve = beta*dzd+icpt
        for j in range(N):
            vel[j] = MOM*vel[j]+VN*np.random.randn(); pos[j] += vel[j]*dm+PN*np.random.randn()
            pos[j] = max(pos[j], vmin-50.); pos[j] = min(pos[j], vmin+len(gg_p)*step+50.)
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                ep = _interp1(gg_p, pos[j], vmin, step); dp = (gr_v[i]-ep)/gs
                lp = max(np.exp(-0.5*dp*dp) if dp*dp < 600. else 0., 1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es = _interp1(gg_s, pos[j], vmin, step); ds = (gr_sm_v[i]-es)/(gs*1.5)
                    lsm = max(np.exp(-0.5*ds*ds) if ds*ds < 600. else 0., 1e-300); lk = (1.-GR_WT)*lp+GR_WT*lsm
                else: lk = lp
                lk = max(lk, 1e-300); w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1./N
        ws2 = 0.
        for j in range(N):
            dv = (vel[j]-ve)/max(zsig*2., 0.005); lz = max(np.exp(-0.5*dv*dv) if dv*dv < 600. else 0., 1e-300)
            w[j] *= lz; ws2 += w[j]
        if ws2 > 0.:
            for j in range(N): w[j] /= ws2
        else:
            for j in range(N): w[j] = 1./N
        ne = 0.
        for j in range(N): ne += w[j]*w[j]
        if 1./ne < RESAMP*N:
            pos, vel = _resamp(pos, vel, w, N, RP, RV)
            for j in range(N): w[j] = 1./N
        wm = 0.
        for j in range(N): wm += w[j]*pos[j]
        pts[i] = wm; va = 0.
        for j in range(N): va += w[j]*(pos[j]-wm)**2
        std_[i] = va**0.5; pm = md_v[i]; pz = z_v[i]
    return pts, std_

# Build a uniform TVT grid and interpolate GR onto it for O(1) lookup
def _grid(tw_tvt, tw_gr, step=0.2):
    tmin = float(tw_tvt.min()); tmax = float(tw_tvt.max())
    tvt_g = np.arange(tmin, tmax+step, step)
    return np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64), float(tmin), float(step)

# Estimate GR observation noise from the known prefix (clipped to bounds)
def _gr_sig(hw, tw_tvt, tw_gr):
    kn = hw[hw.TVT_input.notna() & hw.GR.notna()]
    if len(kn) < 20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn.GR.values-np.interp(kn.TVT_input.values, tw_tvt, tw_gr)),
                         PF_GR_SIG_MIN, PF_GR_SIG_MAX))

# Nearest-index lookup into a sorted array
def _nn(arr, v):
    i = int(np.searchsorted(arr, v, "left"))
    if i >= len(arr): return len(arr)-1
    if i > 0 and abs(arr[i-1]-v) <= abs(arr[i]-v): return i-1
    return i

# Interpolate gaps then optionally rolling-mean smooth a 1-D signal
def _smooth(vals, fb, r):
    s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(fb)
    return (s.rolling(r*2+1, center=True, min_periods=1).mean() if r > 0 else s).to_numpy(np.float32)

# Run one beam config and map the index path back to TVT values
def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si = _nn(tw_tvt, start_tvt); sgr = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    return tw_tvt[_beam_jit(sgr, tw_gr.astype(np.float64), si, bs, float(mc), float(es))].astype(np.float32)

# Prepare inputs and run the ANCC particle filter for a well
def run_pf_ancc(hw, tw_tvt, tw_gr, N=ANCC_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr); kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    ls = float(kn.TVT_input.iloc[-1]+kn.Z.iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail.TVT_input.values); dz = np.diff(tail.Z.values); dm = np.diff(tail.MD.values); m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    pts, std = _pf_ancc(ev.MD.values.astype(np.float64), ev.Z.values.astype(np.float64), ev.GR.values.astype(np.float64),
                        gg, gmin, gst, gs, ls, ir, N, ANCC_ALPHA, ANCC_RN, ANCC_PN, ANCC_IS, ANCC_RP, ANCC_RR, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)

# Prepare inputs (incl. dz/dMD velocity regression) and run the Z particle filter
def run_pf_z(hw, tw_tvt, tw_gr, N=PF_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr); tw_s = pd.Series(tw_gr).rolling(PF_GR_WIN, center=True, min_periods=1).mean().values.astype(np.float32)
    kna = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    dz_k = np.diff(kna.Z.values); dvt = np.diff(kna.TVT_input.values); dmd_k = np.diff(kna.MD.values); m2 = dmd_k > 0
    if m2.sum() >= 10:
        vz = dz_k[m2]/dmd_k[m2]; vt = dvt[m2]/dmd_k[m2]; A = np.column_stack([vz, np.ones_like(vz)])
        c, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
        beta, icpt, zsig = float(c[0]), float(c[1]), max(float(np.std(vt-(c[0]*vz+c[1]))), 0.001)
    else: beta, icpt, zsig = -1., 0., 0.1
    t2 = kna.tail(20); dvt2 = np.diff(t2.TVT_input.values); dmd2 = np.diff(t2.MD.values); m3 = dmd2 > 0
    iv = float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr); gs2, _, _ = _grid(tw_tvt, tw_s)
    gr_sm = hw.GR.rolling(PF_GR_WIN, center=True, min_periods=1).mean()
    pts, std = _pf_z(ev.MD.values.astype(np.float64), ev.Z.values.astype(np.float64), ev.GR.values.astype(np.float64),
                     gr_sm.loc[ev.index].values.astype(np.float64), gg, gs2, gmin, gst, gs,
                     float(kna.TVT_input.iloc[-1]), iv, beta, icpt, zsig, N,
                     PF_MOM, PF_VN, PF_PN, PF_GR_WT, PF_ROUGH_P, PF_ROUGH_V, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)

# Multi-scale normalized cross-correlation -> per-scale signals + softmax ensemble
def multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    out = []
    for hw in hws:
        win = 2*hw+1; nk = len(kgr); nh = len(hgr)
        if nk < win+1 or nh == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        sts = np.arange(0, nk-win+1, stride, dtype=np.int32)
        if len(sts) == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        C = kg[sts[:, None]+np.arange(win, dtype=np.int32)[None, :]].astype(np.float32)
        Cn = (C-C.mean(1, keepdims=True))/(C.std(1, keepdims=True)+1e-6)
        hp = np.pad(hg, hw, mode="edge"); H = hp[np.arange(nh)[:, None]+np.arange(win)[None, :]].astype(np.float32)
        Hn = (H-H.mean(1, keepdims=True))/(H.std(1, keepdims=True)+1e-6)
        ncc = Hn@Cn.T/win; best = ncc.argmax(1); score = ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw, 0, nk-1)].astype(np.float32), score))
    tvts = np.stack([o[0] for o in out], 1); scores = np.stack([o[1] for o in out], 1)
    sw = np.exp(3.*scores); sw /= sw.sum(1, keepdims=True)+1e-9
    return out, (tvts*sw).sum(1).astype(np.float32)

In [ ]:
# ---- 128-seed likelihood-weighted particle filter (the workhorse), numba ---------
# JIT: run the particle filter across many seeds, returning per-seed predictions and log-likelihoods
@njit(cache=True, nogil=True)
def _pf_lik_allseeds(md_v, z_v, gr_v, gg, vmin, step, gs, ls, ir, N, n_seeds, seed_base,
                     MOM, VN, PN, RP, RR, RESAMP, init_spr):
    n = len(md_v); preds = np.empty((n_seeds, n)); liks = np.empty(n_seeds); tmax = vmin + len(gg)*step
    for s in range(n_seeds):
        np.random.seed(seed_base + s)
        pos = np.empty(N); rate = np.empty(N); w = np.ones(N)/N
        for j in range(N):
            pos[j] = ls + init_spr*np.random.randn(); rate[j] = ir + 0.01*np.random.randn()
        log_lik = 0.0; prev_md = md_v[0] - 1.0
        for i in range(n):
            dm = md_v[i] - prev_md
            if dm < 1.0: dm = 1.0
            for j in range(N):
                rate[j] = MOM*rate[j] + VN*np.random.randn(); pos[j] += rate[j]*dm + PN*np.random.randn()
                tvt_j = pos[j] - z_v[i]
                if tvt_j < vmin-100.: tvt_j = vmin-100.
                if tvt_j > tmax+100.: tvt_j = tmax+100.
                pos[j] = tvt_j + z_v[i]
            avg_lk = 0.0
            for j in range(N):
                eg = _interp1(gg, pos[j]-z_v[i], vmin, step); d = (gr_v[i]-eg)/gs; dd = d*d
                if dd > 600.: dd = 600.
                lk = np.exp(-0.5*dd)
                if lk < 1e-300: lk = 1e-300
                avg_lk += w[j]*lk; w[j] = w[j]*lk
            if avg_lk < 1e-300: avg_lk = 1e-300
            log_lik += np.log(avg_lk)
            ws = 0.0
            for j in range(N): ws += w[j]
            if ws > 0.0:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1./N
            neff = 0.0
            for j in range(N): neff += w[j]*w[j]
            neff = 1.0/neff
            if neff < RESAMP*N:
                cum = np.empty(N); c = 0.0
                for j in range(N): c += w[j]; cum[j] = c
                u0 = np.random.uniform(0., 1./N); newpos = np.empty(N); newrate = np.empty(N); ci = 0
                for j in range(N):
                    u = u0 + j/N
                    while ci < N-1 and cum[ci] < u: ci += 1
                    newpos[j] = pos[ci] + RP*np.random.randn(); newrate[j] = rate[ci] + RR*np.random.randn()
                for j in range(N): pos[j] = newpos[j]; rate[j] = newrate[j]; w[j] = 1./N
            est = 0.0
            for j in range(N): est += w[j]*(pos[j]-z_v[i])
            preds[s, i] = est; prev_md = md_v[i]
        liks[s] = log_lik
    return preds, liks

# Likelihood-weighted multi-seed PF ensemble; tempers seeds by evidence at several scales
def lik_pf(hw, tw, n_particles=CFG.PF_PARTICLES, n_seeds=CFG.PF_SEEDS, scales=CFG.PF_SCALES,
           init_spr=4.5, seed_base=0, with_quality=False):
    """Likelihood-weighted PF ensemble. Returns ({pf_scale_X: pred_eval}, ev_index[, quality])."""
    # Build the type-well lookup arrays
    tw_s = tw.sort_values("TVT"); tw_tvt = tw_s.TVT.values.astype(float)
    tw_gr = tw_s.GR.fillna(tw_s.GR.mean()).values.astype(float)
    # Split into known prefix and rows to predict
    kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    if len(ev) == 0: return {}, np.array([]), {}
    last = kn.iloc[-1]; ls = float(last.TVT_input) + float(last.Z)
    tw_at_k = np.interp(kn.TVT_input.values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn.GR.fillna(0).values - tw_at_k), 10., 60.))
    tail = kn.tail(30); dt = np.diff(tail.TVT_input.values); dz = np.diff(tail.Z.values); dm = np.diff(tail.MD.values); m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.0
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    gr_v = hw.GR.interpolate(limit_direction="both").fillna(tw_gr.mean()).values.astype(float)[ev.index]
    preds, liks = _pf_lik_allseeds(ev.MD.values.astype(float), ev.Z.values.astype(float), gr_v,
                                   gg, gmin, gst, gs, ls, ir, n_particles, n_seeds, seed_base,
                                   0.998, 0.002, 0.005, 0.1, 0.001, 0.5, init_spr)
    # Temper the per-seed evidence into ensemble weights per scale
    ln = liks - liks.max(); out = {}
    for sc in scales:
        wts = np.exp(ln/float(sc)); wts /= wts.sum(); out[f"pf_scale_{sc:g}"] = (wts[:, None]*preds).sum(0)
    out["pf_mean"] = preds.mean(0)
    q = {}
    if with_quality:
        q = {"pf_best_ll": float(liks.max())/len(ev), "pf_ll_spread": float(liks.std()),
             "pf_pt_std": preds.std(0).astype(np.float32), "pf_gr_sig": gs}
    return out, ev.index.values, q

# JIT warm-up so timings below are representative
_m = np.linspace(1, 50, 20); _z = np.zeros(20); _g = np.full(20, 50.); _gg = np.linspace(45, 55, 100)
_pf_ancc(_m, _z, _g, _gg, 45., .1, 20., 50., 0., 8, .998, .002, .005, .3, .1, .001, .5)
_pf_z(_m, _z, _g, _g, _gg, _gg, 45., .1, 20., 50., 0., -1., 0., .1, 8, .993, .005, .01, .3, .2, .003, .5)
_beam_jit(np.random.randn(30), np.random.randn(50), 25, 8, 15., 100.)
_pf_lik_allseeds(_m, _z, _g, _gg, 45., .1, 20., 50., 0., 64, 4, 0, .998, .002, .005, .1, .001, .5, 4.5)
print("trackers compiled.")

# Plot single-PF vs 128-seed lik-PF against ground truth for one well
def fig_tracker_vs_truth(wid):
    import matplotlib.pyplot as plt
    hw, tw = load_well(wid); kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    tw_tvt = tw.TVT.to_numpy(np.float32); tw_gr = tw.GR.to_numpy(np.float32); last = float(kn.TVT_input.iloc[-1])
    pf, _ = run_pf_ancc(hw, tw_tvt, tw_gr); out, _, _ = lik_pf(hw, tw, scales=(3.,))
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(ev.MD, ev.TVT, lw=2.2, color="black", label="True TVT", zorder=5)
    ax.plot(ev.MD, np.full(len(ev), last), lw=1.1, color="gray", ls=":", label="last-known baseline")
    ax.plot(ev.MD, pf, lw=1.0, color="tab:blue", alpha=.8, label="single particle filter")
    ax.plot(ev.MD, out["pf_scale_3"], lw=1.5, color="crimson", alpha=.9, label="128-seed lik-weighted PF")
    ax.set_xlabel("MD (ft)"); ax.set_ylabel("TVT (ft)"); ax.invert_yaxis(); ax.grid(alpha=.25)
    ax.set_title(f"Well {wid}: trackers vs ground truth â€” the lik-PF resists drift"); ax.legend(loc="best")
    plt.tight_layout(); plt.show()

In [ ]:
# Show the tracker-vs-truth figure for the demo well when figures are enabled
if CFG.SHOW_FIGS and DEMO:
    fig_tracker_vs_truth(DEMO)

In [ ]:
PLANE_K = 10; DENSE_SPW = 60; DENSE_K = 20

# Outlier-tolerant linear slope of y vs x (0 when ill-conditioned)
def robust_slope(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float); m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2 or np.std(x[m]) < 1e-6: return 0.
    return float(np.polyfit(x[m], y[m], 1)[0])

# Fit an affine GR calibration (slope, offset) of known GR vs type-well GR
def affine_cal(kgr, tw_at_k, min_pts=20):
    v = np.isfinite(kgr) & np.isfinite(tw_at_k)
    if v.sum() < min_pts or np.std(tw_at_k[v]) < 1e-6:
        return 1., float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
    a, b = np.polyfit(tw_at_k[v], kgr[v], 1); return float(a), float(b)

# Segmented stratigraphic offset b_well (full/early/mid/late + tail-weighted)
def seg_b_well(ktvt, kz, form_col):
    bv = ktvt+kz-form_col; n = len(bv); b_full = float(np.median(bv))
    b_late = float(np.median(bv[max(0, n-50):])) if n >= 5 else b_full
    t1, t2 = n//3, 2*n//3
    b_early = float(np.median(bv[:max(1, t1)])) if t1 > 0 else b_full
    b_mid = float(np.median(bv[t1:max(t1+1, t2)])) if t2 > t1 else b_full
    w = np.exp(0.02*np.arange(n)); w /= w.sum()
    return b_full, b_early, b_mid, b_late, float(np.dot(w, bv))

# Spatial plane-fit imputer: predicts each formation depth from nearby wells XY
class FormationPlaneKNN:
    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            try: df = pd.read_csv(data_dir/f"{wid}__horizontal_well.csv", usecols=["X","Y"]+FORMATIONS).dropna()
            except: continue
            if len(df) == 0: continue
            row = {"wid": wid, "x": float(df.X.median()), "y": float(df.Y.median())}
            for c in FORMATIONS: row[f"{c}_m"] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows); self.wmap = {w: i for i, w in enumerate(self.df.wid)}
        xy = self.df[["x","y"]].to_numpy(); self.scale = np.where(xy.std(0) < 1e-3, 1., xy.std(0))
        self.tree = cKDTree(xy/self.scale); self.xa = self.df.x.to_numpy(); self.ya = self.df.y.to_numpy()
        self.fa = self.df[[f"{c}_m" for c in FORMATIONS]].to_numpy(np.float64)
    # Fit a local weighted plane in XY through k neighbours and evaluate at queries
    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q/self.scale; nf = min(k+5, len(self.df)); dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid in self.wmap: dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        ordr = np.argpartition(dist, min(k-1, nf-1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1./(dk+1e-3), 0.).astype(np.float64)
        xn = self.xa[ik]; yn = self.ya[ik]; fn = self.fa[ik]; wx = w*xn; wy = w*yn
        A = np.zeros((len(q), 3, 3))
        A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
        A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
        A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
        A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
        rhs = np.stack([(wx[:,:,None]*fn).sum(1), (wy[:,:,None]*fn).sum(1), (w[:,:,None]*fn).sum(1)], 1)
        try: coef = np.linalg.solve(A, rhs)
        except:
            coef = np.zeros((len(q), 3, 6))
            for r in range(len(q)):
                try: coef[r] = np.linalg.pinv(A[r])@rhs[r]
                except: pass
        Xq = xy_q[:,0]; Yq = xy_q[:,1]
        pred = (Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
        pred[~vk.any(1)] = self.fa.mean(0)
        return pred, np.where(vk, dk, np.inf).min(1).astype(np.float32)

# Dense ANCC imputer: inverse-distance interpolation over sampled points from all wells
class DenseANCCImputer:
    def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
        xs, ys, an, wd = [], [], [], []
        for wid in well_ids:
            try: df = pd.read_csv(data_dir/f"{wid}__horizontal_well.csv", usecols=["X","Y","ANCC"]).dropna()
            except: continue
            if len(df) == 0: continue
            ix = np.linspace(0, len(df)-1, min(spw, len(df)), dtype=int); s = df.iloc[ix]
            xs.append(s.X.values); ys.append(s.Y.values); an.append(s.ANCC.values); wd.extend([wid]*len(s))
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(an).astype(np.float32); self.wids = np.array(wd)
        self.scale = np.where(self.xy.std(0) < 1e-3, 1., self.xy.std(0)); self.tree = cKDTree(self.xy/self.scale)
    # Inverse-distance weighted ANCC estimate plus spread and nearest distance
    def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=5000):
        xy_q = np.atleast_2d(xy_q); q = xy_q/self.scale; nf = min(nfetch, len(self.ancc))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid: dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
        ordr = np.argpartition(dist, min(k-1, nf-1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1./(dk+1e-3), 0.); sw = w.sum(1); safe = np.where(sw < 1e-9, 1., sw)
        a = self.ancc[ik]; ap = (a*w).sum(1)/safe; ap = np.where(sw < 1e-9, float(self.ancc.mean()), ap)
        var = ((a-ap[:,None])**2*w).sum(1)/safe
        return ap.astype(np.float32), np.sqrt(np.maximum(var, 0.)).astype(np.float32), np.where(vk, dk, np.inf).min(1).astype(np.float32)

# Imputer handles (set later) and GR-vs-typewell anchor offset grids
_FI = None; _DI = None
ANCH_OFFS = np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80], np.float32)
BEAM_OFFS = np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40], np.float32)
SC_OFFS = np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30], np.float32)
PF_OFFS = SC_OFFS.copy()

In [ ]:
# Build the full feature row-set for a single well (the heart of feature engineering)
def build_well(hw_path, tw_path, is_train, likpf_map=None):
    global _FI, _DI
    # Recover the well id from the file path
    wid = Path(hw_path).stem.replace("__horizontal_well", "")
    # Load and sort both tables
    try: hw = pd.read_csv(hw_path); tw = pd.read_csv(tw_path).sort_values("TVT")
    except: return None
    if is_train and "TVT" not in hw.columns: return None
    kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    if len(ev) == 0 or len(kn) < 10: return None
    if is_train and hw.TVT.isna().all(): return None
    tw_tvt = tw.TVT.to_numpy(np.float32); tw_gr = tw.GR.to_numpy(np.float32)
    if len(tw_tvt) < 3: return None
    # Particle-filter signals (ANCC + Z variants)
    pf_a, std_a = run_pf_ancc(hw, tw_tvt, tw_gr)
    if len(pf_a) == 0: return None
    pf_z, std_z = run_pf_z(hw, tw_tvt, tw_gr)
    pf_use = pf_a.astype(np.float32); std_use = std_a.astype(np.float32)
    has_z = len(pf_z) == len(pf_a) and not np.any(np.isnan(pf_z))
    lk = kn.iloc[-1]; last_tvt = float(lk.TVT_input)
    # Full horizontal GR with gaps interpolated
    gr_full = hw.GR.astype(float).interpolate(limit_direction="both").fillna(float(np.nanmean(tw_gr)))
    hgr = gr_full.iloc[ev.index[0]:].to_numpy(np.float32); kgr = gr_full.iloc[:len(kn)].to_numpy(np.float32)
    # Run all beam configs and collect results
    bpaths = {tag: beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r) for (bs, mc, es, r, tag) in BEAMS}
    beam_ref = (bpaths["cons"]+bpaths["sm5"])/2.
    # Known TVT for NCC window matching
    ktvt = kn.TVT_input.to_numpy(np.float32)
    # Multi-scale NCC -> per-scale signals + softmax ensemble
    sc_res, sc_ens = multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3)
    sc8, sc8s = sc_res[0]; sc15, sc15s = sc_res[1]; sc25, sc25s = sc_res[2]; sc_cons = (sc8+sc15+sc25)/3.
    # Trust NCC more as prefix grows; blend beam with NCC
    sc_trust = float(np.clip(len(kn)/200., 0., 0.6)); hyb_ref = (1-sc_trust)*beam_ref+sc_trust*sc_ens
    tw_at_k = np.interp(ktvt, tw_tvt, tw_gr).astype(np.float32); a_cal, b_cal = affine_cal(kgr, tw_at_k)
    kmd = kn.MD.to_numpy(np.float32); kz = kn.Z.to_numpy(np.float32)
    pfx_rmse = float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
    slp_all = robust_slope(kmd, ktvt); slp_50 = robust_slope(kmd[-50:], ktvt[-50:]); slp_z = robust_slope(kz, ktvt)
    # Use well id for self-exclusion in spatial imputers (train only)
    swid = wid if is_train else None
    xy_ev = ev[["X","Y"]].to_numpy(np.float64); xy_kn = kn[["X","Y"]].to_numpy(np.float64)
    # Spatial formation imputation at eval and known XY positions
    form_ev, knn_d = _FI.impute(xy_ev, self_wid=swid); form_kn, _ = _FI.impute(xy_kn, self_wid=swid)
    z_kn = kn.Z.to_numpy(np.float32); z_ev = ev.Z.to_numpy(np.float32)
    # Per-formation: segment b_well + TVT + known-zone RMSE
    tvt_fs = {}; form_rmse = {}; form_list = []
    for fi2, fn in enumerate(FORMATIONS):
        b_full, b_early, b_mid, b_late, b_wls = seg_b_well(ktvt, z_kn, form_kn[:, fi2])
        tvt_f = (-z_ev+form_ev[:, fi2]+b_full).astype(np.float32)
        tvt_fs[f"tvtF_{fn}"]=tvt_f; tvt_fs[f"tvtFw_{fn}"]=(-z_ev+form_ev[:,fi2]+b_wls).astype(np.float32)
        tvt_fs[f"tvtF50_{fn}"]=(-z_ev+form_ev[:,fi2]+b_late).astype(np.float32)
        tvt_fs[f"bw_{fn}"]=np.float32(b_full); tvt_fs[f"bww_{fn}"]=np.float32(b_wls); tvt_fs[f"bw50_{fn}"]=np.float32(b_late)
        tvt_fs[f"bw_early_{fn}"]=np.float32(b_early); tvt_fs[f"bw_mid_{fn}"]=np.float32(b_mid)
        form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2))); form_list.append(tvt_f)
    fs = np.stack(form_list, 1)
    form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32); form_std_d=fs.std(1).astype(np.float32); form_rng_d=(fs.max(1)-fs.min(1)).astype(np.float32)
    # Dense-ANCC imputation and its derived TVT estimates
    d_ancc, d_std, d_dist = _DI.impute(xy_ev, self_wid=swid); d_kn, d_std_kn, _ = _DI.impute(xy_kn, self_wid=swid)
    _, b_de, b_dm, b_dl, b_dw = seg_b_well(ktvt, z_kn, d_kn); b_d = float(np.median(ktvt+z_kn-d_kn))
    tvt_dense=(-z_ev+d_ancc+b_d).astype(np.float32); tvt_densew=(-z_ev+d_ancc+b_dw).astype(np.float32); tvt_dense50=(-z_ev+d_ancc+b_dl).astype(np.float32)
    res_kn = ktvt+z_kn-d_kn; d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))
    # Cross-signal agreement (std/mean) across all candidate TVT signals
    all_sigs=[pf_use]+list(bpaths.values())+[sc8,sc15,sc25,sc_ens,tvt_fs["tvtF_ANCC"],tvt_dense]
    sig_mat=np.stack(all_sigs,1); sig_std=sig_mat.std(1).astype(np.float32); sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)
    # Rolling GR statistics, lags/leads, derivatives, envelope and energy
    gr_s=pd.Series(gr_full.values); rolls={}
    for w in [5,21,51,101]:
        r=gr_s.rolling(w,center=True,min_periods=1); rolls[f"grm{w}"]=r.mean().iloc[ev.index].values.astype(np.float32); rolls[f"grs{w}"]=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1,5,15,30]:
        rolls[f"glag{lag}"]=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32); rolls[f"glead{lag}"]=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32); gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)).iloc[ev.index].values.astype(np.float32)
    # Slope-extrapolated TVT baselines from the last-known point
    hmd=ev.MD.to_numpy(np.float32); md_since=hmd-float(lk.MD)
    slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32); slp_b_50=(last_tvt+slp_50*md_since).astype(np.float32)
    # Geometric per-step gradients (dZ/dMD, dX/dMD, dY/dMD)
    mdd=hw.MD.diff().replace(0,np.nan)
    dzdmd=(hw.Z.diff()/mdd).iloc[ev.index].values.astype(np.float32); dxdmd=(hw.X.diff()/mdd).iloc[ev.index].values.astype(np.float32); dydmd=(hw.Y.diff()/mdd).iloc[ev.index].values.astype(np.float32)
    nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
    def sc(v): return np.full(nh, np.float32(v), np.float32)
    feats={"well":wid,"id":[f"{wid}_{i}" for i in ev.index],"last_known_tvt":sc(last_tvt),
        "pf_ancc":pf_use,"pf_ancc_std":std_use,"pf_ancc_delta":(pf_use-last_tvt).astype(np.float32),
        "pf_z":(pf_z.astype(np.float32) if has_z else sc(last_tvt)),"pf_z_delta":((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
        "pf_vs_z":((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
        **{f"beam_{t}_d":(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
        "beam_mean_d":np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
        "beam_std_d":np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
        "beam_med_d":np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
        "sc8_d":(sc8-np.float32(last_tvt)).astype(np.float32),"sc8_sc":sc8s,"sc15_d":(sc15-np.float32(last_tvt)).astype(np.float32),"sc15_sc":sc15s,
        "sc25_d":(sc25-np.float32(last_tvt)).astype(np.float32),"sc25_sc":sc25s,"sc_cons_d":(sc_cons-np.float32(last_tvt)).astype(np.float32),
        "sc_ens_d":(sc_ens-np.float32(last_tvt)).astype(np.float32),"sc_trust":sc(sc_trust),"hyb_d":(hyb_ref-np.float32(last_tvt)).astype(np.float32),
        "sig_std":sig_std,"sig_mean_d":sig_mean,**tvt_fs,**{f"frm_rmse_{fn}":sc(form_rmse[fn]) for fn in FORMATIONS},
        "form_mean_d":form_mean_d,"form_std_d":form_std_d,"form_rng_d":form_rng_d,
        "spatial_ancc_d":(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),"spatial_knn_dist":knn_d,
        "dense_ancc":d_ancc,"dense_std":d_std,"dense_dist":d_dist,"tvt_dense_d":(tvt_dense-last_tvt).astype(np.float32),
        "tvt_densew_d":(tvt_densew-last_tvt).astype(np.float32),"tvt_dense50_d":(tvt_dense50-last_tvt).astype(np.float32),
        "dense_rmse":sc(d_rmse),"dense_bias":sc(d_bias),"dense_nb_std":sc(d_nb_std),
        "pf_vs_spatial":(pf_use-tvt_fs["tvtF_ANCC"]).astype(np.float32),"pf_vs_dense":(pf_use-tvt_dense).astype(np.float32),
        "spatial_vs_dense":(tvt_fs["tvtF_ANCC"]-tvt_dense).astype(np.float32),"beam_vs_spatial":(bpaths["cons"]-tvt_fs["tvtF_ANCC"]).astype(np.float32),
        "sc_vs_beam":(sc_ens-bpaths["cons"]).astype(np.float32),"cal_a":sc(a_cal),"cal_b":sc(b_cal),
        "pfx_rmse":sc(pfx_rmse),"known_len":sc(len(kn)),"eval_len":sc(nh),"slp_all":sc(slp_all),"slp_50":sc(slp_50),"slp_z":sc(slp_z),
        "slp_b_d_all":(slp_b_all-last_tvt).astype(np.float32),"slp_b_d_50":(slp_b_50-last_tvt).astype(np.float32),
        "ktvt_range":sc(float(np.ptp(ktvt))),"ktvt_std":sc(float(ktvt.std())),"md_since":md_since,"frac":frac,"frac2":frac**2,"sqrt_frac":np.sqrt(frac),
        "z":z_ev,"dx":(ev.X-float(lk.X)).to_numpy(np.float32),"dy":(ev.Y-float(lk.Y)).to_numpy(np.float32),"dz":(z_ev-float(lk.Z)).astype(np.float32),
        "dxy":np.sqrt((ev.X-float(lk.X))**2+(ev.Y-float(lk.Y))**2).to_numpy(np.float32),"dzdmd":dzdmd,"dxdmd":dxdmd,"dydmd":dydmd,
        "gr":hgr,"gr_d1":gr_d1,"gr_d2":gr_d2,"gr_env":gr_env,"gr_nrg":gr_nrg,
        "gr_vs_tw_anc":hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),"gr_vs_slp_all":hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
        **{f"tda{int(o)}":hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
        **{f"tdbc{int(o)}":hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f"tdsc{int(o)}":hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
        **{f"tdpf{int(o)}":hgr-np.interp(pf_use+o,tw_tvt,tw_gr).astype(np.float32) for o in PF_OFFS},
        "tw_range":sc(float(np.ptp(tw_tvt))),"tw_gr_mean":sc(float(tw_gr.mean()))}
    # Merge in the rolling-GR features
    for k,v in rolls.items(): feats[k]=v
    # Create the DataFrame from the feature dict
    res = pd.DataFrame(feats)
    # For training, attach the regression target
    if is_train: res["target"]=(ev.TVT.to_numpy(np.float32)-np.float32(last_tvt))
    return res

# Build the spatial imputers from training wells
def init_imputers(train_wids):
    global _FI, _DI
    _FI = FormationPlaneKNN(train_wids, CFG.DATA/"train"); _DI = DenseANCCImputer(train_wids, CFG.DATA/"train")

# Run the 128-seed likelihood-weighted particle-filter ensemble for one well
def _likpf_rows(wid, split):
    hw, tw = load_well(wid, split)
    out, idx, _ = lik_pf(hw, tw)
    if not len(out): return None
    d = {"id": [f"{wid}_{i}" for i in idx]}
    for k, v in out.items():
        d["likpf_" + k.replace("pf_scale_", "scale_").replace("pf_mean", "mean")] = v.astype(np.float32)
    return pd.DataFrame(d)

# Build lik-PF features in parallel across wells (threads safe: nogil=True)
def build_likpf(wids, split):
    # threads are safe here: the lik-PF numba kernel is compiled with nogil=True, so it
    # releases the GIL and parallelises across threads (no pickling of numba code needed).
    res = Parallel(n_jobs=CFG.n_jobs, prefer="threads")(delayed(_likpf_rows)(w, split) for w in wids)
    return pd.concat([r for r in res if r is not None], ignore_index=True)

# Build all features in parallel for a list of wells
def build_features(wids, split, is_train):
    paths = [CFG.DATA/split/f"{w}__horizontal_well.csv" for w in wids]
    res = Parallel(n_jobs=CFG.n_jobs, prefer="threads")(
        delayed(build_well)(str(p), str(p.parent/f"{p.stem.replace('__horizontal_well','')}__typewell.csv"), is_train)
        for p in paths if (p.parent/f"{p.stem.replace('__horizontal_well','')}__typewell.csv").exists())
    parts = [r for r in res if r is not None]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

# Merge lik-PF predictions into the feature DataFrame and create delta columns
def add_likpf_features(df, likpf):
    df = df.merge(likpf, on="id", how="left")
    for c in [c for c in likpf.columns if c != "id"]:
        df[c] = df[c].fillna(df["last_known_tvt"]); df[c+"_d"] = (df[c]-df["last_known_tvt"]).astype(np.float32)
    return df

In [ ]:
# Detect GPU availability (nvidia-smi) or fall back to CPU
def _device():
    if CFG.USE_GPU == "cpu": return "cpu", "CPU"
    if CFG.USE_GPU == "gpu": return "gpu", "GPU"
    try:  # detect a real NVIDIA GPU (Kaggle GPU accelerator) via nvidia-smi
        import subprocess
        # Check for NVIDIA GPU
        if subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0:
            return "gpu", "GPU"
    except Exception:
        pass
    return "cpu", "CPU"

# Return a list of LightGBM hyper-parameter sets tuned for the device
def lgb_configs(dev):
    base = dict(boosting_type="gbdt", objective="regression", verbose=-1, n_jobs=-1, max_bin=255)
    if dev == "gpu": base.update(device_type="gpu", gpu_use_dp=False)
    n = 600 if CFG.FAST else 5000
    return [
        dict(**base, num_leaves=255, min_child_samples=15, subsample=0.8, subsample_freq=1,
             colsample_bytree=0.8, reg_lambda=3.0, reg_alpha=0.05, learning_rate=0.03, n_estimators=n, seed=123),
        dict(**base, num_leaves=64, min_child_samples=40, subsample=0.474, subsample_freq=1,
             colsample_bytree=0.393, reg_lambda=95.75, reg_alpha=10.79, min_child_weight=0.24,
             learning_rate=0.0093, n_estimators=min(2*n, 10000), random_state=0),
        dict(**base, num_leaves=64, min_child_samples=40, subsample=0.474, subsample_freq=1,
             colsample_bytree=0.393, reg_lambda=95.75, reg_alpha=10.79, min_child_weight=0.24,
             learning_rate=0.0093, n_estimators=min(2*n, 10000), random_state=29),
    ]

# Return a list of CatBoost hyper-parameter sets tuned for the device
def cb_configs(dev):
    tt = "GPU" if dev == "gpu" else "CPU"
    n = 800 if CFG.FAST else 8000
    return [
        dict(iterations=n, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254,
             loss_function="RMSE", task_type=tt, od_type="Iter", od_wait=300, verbose=0, learning_rate=0.02, random_seed=7),
        dict(iterations=n, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254,
             loss_function="RMSE", task_type=tt, od_type="Iter", od_wait=300, verbose=0, learning_rate=0.03, random_seed=123),
    ]

# Train an ensemble stack: LightGBM + CatBoost base models + Ridge meta-learner
def train_stack(train_df, test_df, features):
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
    from catboost import CatBoostRegressor
    from sklearn.model_selection import GroupKFold
    from sklearn.linear_model import Ridge
    # Detect and report the device (GPU/CPU)
    dev, devname = _device(); print("device:", devname)
    X = train_df[features].values.astype(np.float32); y = train_df["target"].values.astype(np.float32)
    g = train_df["well"].values; Xt = test_df[features].values.astype(np.float32)
    # Set up group-aware cross-validation and result dicts
    cv = GroupKFold(CFG.n_splits); oof_cols = {}; test_cols = {}
    # Inner function: train and predict one model across CV folds
    def run(name, make, fit_kw, is_lgb):
        # LightGBM: slice to best_iteration_ via num_iteration. CatBoost: use_best_model
        # already trims to the best tree, and its predict() takes no num_iteration kwarg.
        oof = np.zeros(len(train_df)); tp = np.zeros(len(test_df))
        for tr, va in cv.split(X, y, groups=g):
            m = make(); m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], **fit_kw)
            if is_lgb:
                it = m.best_iteration_
                # LightGBM: use best_iteration_ to trim predictions
                oof[va] = m.predict(X[va], num_iteration=it); tp += m.predict(Xt, num_iteration=it) / CFG.n_splits
            else:
                # CatBoost: use_best_model already trims to best
                oof[va] = m.predict(X[va]); tp += m.predict(Xt) / CFG.n_splits
        oof_cols[name] = oof; test_cols[name] = tp
        print(f"  {name}: OOF RMSE={rmse(y, oof):.4f}", flush=True)
    for i, p in enumerate(lgb_configs(dev)):
        run(f"lgb{i}", lambda p=p: LGBMRegressor(**p),
            dict(eval_metric="rmse", callbacks=[early_stopping(250, verbose=False), log_evaluation(0)]), True)
    for i, p in enumerate(cb_configs(dev)):
        run(f"cb{i}", lambda p=p: CatBoostRegressor(**p),
            dict(early_stopping_rounds=250, use_best_model=True), False)
    # Stack base-model predictions into DataFrames
    OOF = pd.DataFrame(oof_cols); TEST = pd.DataFrame(test_cols)
    # Ridge meta-learner with non-negative weights
    rid = Ridge(alpha=1.66, positive=True, fit_intercept=True); meta = np.zeros(len(train_df))
    for tr, va in cv.split(OOF.values, y, groups=g):
        # Fit Ridge on the train fold and predict on the val fold
        rid.fit(OOF.values[tr], y[tr]); meta[va] = rid.predict(OOF.values[va])
    rid.fit(OOF.values, y); meta_test = rid.predict(TEST.values)
    print(f"  ridge-stack OOF RMSE={rmse(y, meta):.4f}")
    return meta, meta_test, OOF, TEST

In [ ]:
class PP:   # tuned on 773-well GroupKFold OOF (Nelder-Mead + grid; the optimum is flat)
    alpha = 1.0         # global scale on the learned delta (tuned ~1.0)
    tau = 85.0          # warm-up length in ft: damps the first feet after PS (tuned ~90)
    w_pf = 0.0          # blending the model with the single PF no longer helps once lik-PF is a feature
    w_sub1 = 0.60       # weight on the learned model; lik-PF gets 1-w_sub1. CV optimum ~0.68 (flat
                        # 0.55-0.68); 0.60 is a small hedge toward the drift-robust lik-PF for LB transfer.
    sub2_scale = "scale_5"   # which likelihood-scale of the lik-PF to use as sub2 (3/5/8 ~equivalent)
    sg_win = 61         # per-well Savitzky-Golay smoothing window (effect is small, ~0.01 ft)
    sg_poly = 3

# Ramp the delta up over MD-since-last-known
def warmup(md_since, tau): return 1.-np.exp(-np.maximum(md_since, 0.)/tau) if tau > 1e-6 else 1.0

# Blend model delta with lik-PF, ramp by warmup, and smooth per-well
def make_prediction(df, model_delta, likpf):
    # Last-known TVT baseline per row
    last = df["last_known_tvt"].values.astype(float)
    # Particle-filter delta relative to the baseline
    pf_delta = df["pf_ancc"].values.astype(float) - last
    # Likelihood-PF delta at the chosen scale
    lp = df[f"likpf_{PP.sub2_scale}"].values.astype(float) - last
    # Post-processed stacked delta with blend and warmup
    sub1 = PP.alpha*warmup(df["md_since"].values.astype(float), PP.tau)*(model_delta*(1-PP.w_pf)+pf_delta*PP.w_pf)
    # Final delta: blend sub1 model with lik-PF
    delta = PP.w_sub1*sub1 + (1-PP.w_sub1)*lp
    # Absolute prediction
    pred = last + delta
    # per-well Savitzky-Golay smoothing
    out = pred.copy(); dfx = df.reset_index(drop=True)
    # Smooth each well independently
    for _, idx in dfx.groupby("well", sort=False).groups.items():
        pos = dfx.index.get_indexer(idx); v = pred[pos]; n = len(v); wl = min(PP.sg_win, n)
        if wl % 2 == 0: wl -= 1
        if wl >= PP.sg_poly+2: out[pos] = savgol_filter(v, wl, PP.sg_poly)
    return out

In [ ]:
# Look for mounted pre-trained boosters (inference mode) or return None (train from scratch)
def _find_models():
    """Look for a mounted dataset of pre-trained boosters (lgb*.pkl + features.json).
    If present we run in fast INFERENCE mode; otherwise we train from scratch."""
    import glob as _g
    for f in _g.glob("/kaggle/input/**/features.json", recursive=True):
        d = Path(f).parent
        if list(d.glob("lgb*.pkl")):
            return d
    d = CFG.OUT / "models"
    return d if (d/"features.json").exists() and list(d.glob("lgb*.pkl")) else None

# Self-contained orchestration: build features, train stack, make submission
def main():
    import json, joblib, glob as _g
    t0 = time.time()
    # Enumerate training wells
    train_wids = sorted(p.stem.replace("__horizontal_well", "") for p in (CFG.DATA/"train").glob("*__horizontal_well.csv"))
    # Enumerate test wells
    test_wids = sorted(p.stem.replace("__horizontal_well", "") for p in (CFG.DATA/"test").glob("*__horizontal_well.csv"))
    if CFG.N_TRAIN_WELLS: train_wids = train_wids[:CFG.N_TRAIN_WELLS]
    print(f"train wells: {len(train_wids)} | test wells: {len(test_wids)}")
    init_imputers(train_wids)   # offset-well spatial priors are built from the train wells

    # --- test features are always computed dynamically (works on the hidden test set) ---
    print("building lik-PF + features (test)â€¦", flush=True)
    likpf_test = build_likpf(test_wids, "test")
    test_df = add_likpf_features(build_features(test_wids, "test", is_train=False), likpf_test).reset_index(drop=True)

    # Check for pre-trained models on the input mount
    models_dir = _find_models()
    cv_final = None
    # Fast INFERENCE: load and apply pre-trained boosters
    if models_dir is not None:
        # ---------- fast INFERENCE: load pre-trained boosters ----------
        print(f"INFERENCE mode â€” loading models from {models_dir}", flush=True)
        feats = json.load(open(models_dir/"features.json"))
        models = [joblib.load(p) for p in sorted(models_dir.glob("lgb*.pkl"))]
        for c in feats:
            if c not in test_df.columns: test_df[c] = 0.0
        Xt = test_df[feats].values.astype(np.float32)
        meta_test = np.mean([m.predict(Xt) for m in models], axis=0)
        fallback = float(test_df["last_known_tvt"].mean())
    # Full TRAIN from scratch (self-contained, reproducible)
    else:
        # ---------- full TRAIN from scratch (self-contained, reproducible) ----------
        print("building lik-PF (train)â€¦", flush=True)
        likpf_train = build_likpf(train_wids, "train")
        print("building features (train)â€¦", flush=True)
        train_df = add_likpf_features(build_features(train_wids, "train", is_train=True), likpf_train)
        feats = [c for c in train_df.columns if c not in {"well", "id", "target"}
                 and not (c.startswith("likpf_scale_") or c == "likpf_mean") and c in test_df.columns]
        print(f"features: {len(feats)} | train rows: {len(train_df)} | test rows: {len(test_df)}")
        meta_oof, meta_test, OOF, TEST = train_stack(train_df, test_df, feats)
        y = train_df["target"].values.astype(float)
        cv_final = rmse(train_df["last_known_tvt"].values + y, make_prediction(train_df, meta_oof, None))
        print(f"\n*** tuned CV pooled-RMSE (TVT) = {cv_final:.4f} ***")
        fallback = float(train_df["last_known_tvt"].mean() + y.mean())

    # --- drift-aware blend + submission ---
    # Post-process test predictions and apply per-well smoothing
    test_pred = make_prediction(test_df, meta_test, None)
    # Read the submission template
    sub = pd.read_csv(CFG.DATA/"sample_submission.csv")
    # Map predictions to submission ids with a fallback estimate
    sub["tvt"] = sub["id"].map(dict(zip(test_df["id"], test_pred))).fillna(fallback)
    sub.to_csv(CFG.OUT/"submission.csv", index=False)
    print(f"submission.csv written ({len(sub)} rows) in {time.time()-t0:.0f}s")
    return sub, cv_final

sub, cv_final = main()
sub.head()

In [ ]:
# Plot the ablation study: performance vs baselines
def fig_results():
    import matplotlib.pyplot as plt
    names = ["last-known", "LGBM (orig. feats)", "stack + lik-PF feats", "baseline recipe", "ours (final)"]
    vals = [15.91, 10.85, 9.69, 9.75, cv_final if cv_final else 9.21]
    colors = ["#bbb", "#7aa", "#5a8", "#caa", "crimson"]
    fig, ax = plt.subplots(figsize=(9, 4))
    b = ax.barh(names[::-1], vals[::-1], color=colors[::-1])
    for r, v in zip(b, vals[::-1]): ax.text(v+0.1, r.get_y()+r.get_height()/2, f"{v:.2f}", va="center")
    ax.set_xlabel("CV pooled-RMSE (ft, lower is better)"); ax.set_title("Ablation â€” GroupKFold CV")
    ax.grid(alpha=.25, axis="x"); plt.tight_layout(); plt.show()

# Show results only when figures are enabled
if CFG.SHOW_FIGS:
    fig_results()

In [ ]:
# Final multi-approach blend: sp45 projection + fleongg inference
from pathlib import Path as _FinalBlendPath
import numpy as _final_np
import pandas as _final_pd

_WORK = _FinalBlendPath('/kaggle/working') if _FinalBlendPath('/kaggle/working').exists() else _FinalBlendPath('.')
# Test a range of sp45 weights to find the best blend
_BLEND_WEIGHTS_SP45 = (0.50, 0.52, 0.55, 0.58, 0.60)
# Selected blend weight (55% sp45, 45% fleongg)
_SELECTED_SP45_WEIGHT = 0.55
# Input submission files to blend
_INPUT_FILES = {
    'fleongg': _WORK / 'submission.csv',
    'sp45': _WORK / 'sp45_projection_submission.csv',
}


# Read and validate a submission file
def _read_submission_frame(path, label):
    frame = _final_pd.read_csv(path)
    missing = {'id', 'tvt'} - set(frame.columns)
    if missing:
        raise RuntimeError(f'{label} submission is missing columns: {sorted(missing)}')

    frame = frame[['id', 'tvt']].copy()
    frame['id'] = frame['id'].astype(str)
    frame['tvt'] = frame['tvt'].astype(float)

    if not _final_np.isfinite(frame['tvt'].to_numpy(dtype=float)).all():
        raise RuntimeError(f'Non-finite values in {label} tvt')
    return frame


# Merge two submissions on id (verify all ids match)
def _merge_blend_inputs(sp45, fleongg):
    merged = sp45.rename(columns={'tvt': 'tvt_sp45'}).merge(
        fleongg.rename(columns={'tvt': 'tvt_fleongg'}),
        on='id',
        how='inner',
    )
    if len(merged) != len(sp45) or len(merged) != len(fleongg):
        raise RuntimeError(
            f'Blend id mismatch: sp45={len(sp45)}, fleongg={len(fleongg)}, merged={len(merged)}'
        )
    return merged


# Create a weighted-average submission
def _weighted_submission(merged, w_sp45):
    w_fleongg = 1.0 - float(w_sp45)
    out = merged[['id']].copy()
    out['tvt'] = (
        float(w_sp45) * merged['tvt_sp45'].astype(float)
        + w_fleongg * merged['tvt_fleongg'].astype(float)
    )
    return out


# Build a statistics row for a candidate submission
def _candidate_report_row(candidate, merged, file_name, w_sp45):
    diff = candidate['tvt'].to_numpy(dtype=float) - merged['tvt_sp45'].to_numpy(dtype=float)
    return {
        'file': file_name,
        'w_sp45': float(w_sp45),
        'w_fleongg': float(1.0 - w_sp45),
        'rows': int(len(candidate)),
        'mean_tvt': float(candidate['tvt'].mean()),
        'std_tvt': float(candidate['tvt'].std()),
        'rmse_vs_sp45': float(_final_np.sqrt(_final_np.mean(diff * diff))),
        'p95_abs_vs_sp45': float(_final_np.quantile(_final_np.abs(diff), 0.95)),
    }


_fle = _read_submission_frame(_INPUT_FILES['fleongg'], 'fleongg')
_fle.to_csv(_WORK / 'fleongg_pretrained_submission.csv', index=False)
_sp45 = _read_submission_frame(_INPUT_FILES['sp45'], 'sp45')
_merged = _merge_blend_inputs(_sp45, _fle)

_report_rows = []
for _w_sp45 in _BLEND_WEIGHTS_SP45:
    _candidate = _weighted_submission(_merged, _w_sp45)
    _name = f'submission_sp45_fleongg_w{_w_sp45:.2f}.csv'
    _candidate.to_csv(_WORK / _name, index=False)
    _report_rows.append(_candidate_report_row(_candidate, _merged, _name, _w_sp45))

_final_name = f'submission_sp45_fleongg_w{_SELECTED_SP45_WEIGHT:.2f}.csv'
_final = _final_pd.read_csv(_WORK / _final_name)
_final.to_csv(_WORK / 'submission.csv', index=False)

_report = _final_pd.DataFrame(_report_rows)
_report.to_csv(_WORK / 'sp45_fleongg_blend_report.csv', index=False)
print(_report.to_string(index=False), flush=True)
print('wrote final submission.csv from', _final_name, _final.shape, flush=True)

In [ ]:
# Lesson learned: hidden rerun copies of "overlap" wells are NOT guaranteed to be
# same-version / row-aligned with their train copies - a blind 100% lookup can inject error.
# Guard: per well, validate the contacts reconstruction against the TEST copy's known
# prefix (TVT_input), interpolated BY MD (not row index); override only if rmse < 1 ft,
# and only rows whose MD lies inside the train copy's range. Otherwise keep the blend.
# By construction this is >= the plain blend: exact wells win, mismatched wells are skipped.
# Override logic: validate physical model against the test copy
import os as _ov_os, glob as _ov_glob
import numpy as _ov_np, pandas as _ov_pd
from pathlib import Path as _OvPath

# Derive TVT from formation contacts (physical model)
def _ov_tvt_from_contacts(hw_tr, tw_tr, ref_col="EGFDU"):
    tw_g = tw_tr.dropna(subset=["Geology"])
    ref_tvt = tw_g[tw_g["Geology"] == ref_col]["TVT"].min()
    if _ov_np.isnan(ref_tvt):
        ref_col = tw_g["Geology"].iloc[0]; ref_tvt = tw_g[tw_g["Geology"] == ref_col]["TVT"].min()
    offset = (hw_tr["TVT"] - (ref_tvt - (hw_tr["Z"] - hw_tr[ref_col]))).mean()
    return (ref_tvt - (hw_tr["Z"] - hw_tr[ref_col]) + offset).to_numpy(dtype=float)

try:
    _W = _OvPath("/kaggle/working") if _OvPath("/kaggle/working").exists() else _OvPath(".")
    _DATA = None
    for _c in [_OvPath("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
               _OvPath("/kaggle/input/rogii-wellbore-geology-prediction")]:
        if _c.exists() and (_c / "train").exists():
            _DATA = _c; break
    if _DATA is None:
        for _p in _ov_glob.glob("/kaggle/input/**/train/*__horizontal_well.csv", recursive=True):
            _DATA = _OvPath(_p).parent.parent; break
    _sub = _ov_pd.read_csv(_W / "submission.csv")
    _sub["well"] = _sub["id"].str[:8]; _sub["row_idx"] = _sub["id"].str[9:].astype(int)
    _pred = dict(zip(_sub["id"].astype(str), _sub["tvt"].astype(float)))
    _train_wells = set(_ov_os.path.basename(f).split("__")[0]
                       for f in _ov_glob.glob(str(_DATA / "train" / "*__horizontal_well.csv")))
    _n_ok = _n_skip = 0
    for _wid, _g in _sub.groupby("well"):
        if _wid not in _train_wells:
            continue
        try:
            _hw_te = _ov_pd.read_csv(_DATA / "test" / (_wid + "__horizontal_well.csv"))
            _hw_tr = _ov_pd.read_csv(_DATA / "train" / (_wid + "__horizontal_well.csv"))
            _tw_tr = _ov_pd.read_csv(_DATA / "train" / (_wid + "__typewell.csv"))
            _phys = _ov_tvt_from_contacts(_hw_tr, _tw_tr)
            _md_raw = _hw_tr["MD"].to_numpy(dtype=float)
            _m_fin = _ov_np.isfinite(_phys) & _ov_np.isfinite(_md_raw)
            if _m_fin.sum() < 100:
                print("override SKIP %s too few valid phys rows=%d" % (_wid, int(_m_fin.sum()))); _n_skip += 1; continue
            _o = _ov_np.argsort(_md_raw[_m_fin])
            _md_tr = _md_raw[_m_fin][_o]; _ph_tr = _phys[_m_fin][_o]
            # --- self-check: TEST copy known prefix (TVT_input) vs lookup, interpolated by MD ---
            _kn = _hw_te[_hw_te["TVT_input"].notna()]
            _kn = _kn[(_kn["MD"] >= _md_tr[0]) & (_kn["MD"] <= _md_tr[-1])]
            if len(_kn) < 50:
                print("override SKIP %s too few comparable known-prefix rows=%d" % (_wid, len(_kn))); _n_skip += 1; continue
            _rk = float(_ov_np.sqrt(_ov_np.mean(
                (_ov_np.interp(_kn["MD"].to_numpy(dtype=float), _md_tr, _ph_tr)
                 - _kn["TVT_input"].to_numpy(dtype=float)) ** 2)))
            if (not _ov_np.isfinite(_rk)) or _rk > 1.0:
                print("override SKIP %s known-prefix rmse=%.3f (train copy != test copy, keeping blend)" % (_wid, _rk))
                _n_skip += 1; continue
            # --- check passed -> override via MD interpolation (no row-index alignment), in-range rows only ---
            _md_te = _hw_te["MD"].to_numpy(dtype=float)
            _n_row = 0
            for _rid, _ri in zip(_g["id"].astype(str).values, _g["row_idx"].values):
                _ri = int(_ri)
                if 0 <= _ri < len(_md_te):
                    _m = float(_md_te[_ri])
                    if _md_tr[0] <= _m <= _md_tr[-1]:
                        _pred[_rid] = float(_ov_np.interp(_m, _md_tr, _ph_tr)); _n_row += 1
            print("override OK   %s known-prefix rmse=%.4f rows overridden=%d/%d" % (_wid, _rk, _n_row, len(_g)))
            _n_ok += 1
        except Exception as _e:
            print("override fallback %s: %s" % (_wid, _e)); _n_skip += 1
    _new = _sub["id"].astype(str).map(_pred).astype(float)
    assert _new.notna().all(), "override produced NaN, aborting"
    _sub["tvt"] = _new
    _sub[["id", "tvt"]].to_csv(_W / "submission.csv", index=False)
    print("GUARDED override done: overridden=%d skipped=%d (skipped = kept the blend)" % (_n_ok, _n_skip))
except Exception as _e:
    print("GUARDED override skipped entirely (kept the blend):", _e)

In [ ]:
# Version 2 artifact audit: verify the final file without changing predictions.
# Audit the final submission file for integrity and metadata
import hashlib as _audit_hashlib
import json as _audit_json
import pandas as _audit_pd
import numpy as _audit_np
from pathlib import Path as _AuditPath

_AUDIT_WORK = _AuditPath('/kaggle/working') if _AuditPath('/kaggle/working').exists() else _AuditPath('.')
_AUDIT_SUBMISSION = _AUDIT_WORK / 'submission.csv'

# CFG was redefined in the fleongg section (using .DATA instead of .dataset_path)
_AUDIT_DATA = getattr(CFG, 'DATA', getattr(CFG, 'dataset_path', _AuditPath('/kaggle/input/competitions/rogii-wellbore-geology-prediction')))
_AUDIT_SAMPLE = _AUDIT_DATA / 'sample_submission.csv'


# Compute SHA256 hash of a file
def _sha256_file(path):
    h = _audit_hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


# Validate and audit the submission against the sample template
def _build_submission_audit(sub_path, sample_path):
    sub = _audit_pd.read_csv(sub_path)
    sample = _audit_pd.read_csv(sample_path)
    if list(sub.columns) != ['id', 'tvt']:
        raise RuntimeError(f'Unexpected submission columns: {list(sub.columns)}')
    if len(sub) != len(sample):
        raise RuntimeError(f'Unexpected row count: submission={len(sub)} sample={len(sample)}')
    if not sub['id'].astype(str).equals(sample['id'].astype(str)):
        raise RuntimeError('Submission id order does not match sample_submission.csv')
    tvt = sub['tvt'].to_numpy(dtype=float)
    if not _audit_np.isfinite(tvt).all():
        raise RuntimeError('Submission contains non-finite tvt values')
    return {
        'rows': int(len(sub)),
        'columns': list(sub.columns),
        'id_order_matches_sample': True,
        'tvt_min': float(_audit_np.min(tvt)),
        'tvt_max': float(_audit_np.max(tvt)),
        'tvt_mean': float(_audit_np.mean(tvt)),
        'tvt_std': float(_audit_np.std(tvt)),
        'sha256_submission_csv': _sha256_file(sub_path),
    }


# Run the submission audit
_audit = _build_submission_audit(_AUDIT_SUBMISSION, _AUDIT_SAMPLE)
with open(_AUDIT_WORK / 'version2_submission_audit.json', 'w', encoding='utf-8') as f:
    _audit_json.dump(_audit, f, indent=2, sort_keys=True)

# Keep a named copy for manual inspection; Kaggle still submits submission.csv.
_audit_pd.read_csv(_AUDIT_SUBMISSION).to_csv(_AUDIT_WORK / 'submission_version2_refactor_copy.csv', index=False)
print('Version 2 submission audit:', _audit, flush=True)

In [ ]:

# Gold visible-prefix calibration overlay.
# It runs AFTER the public final blend and guarded contact override.
# The public submission remains the anchor; this layer only makes a per-well move
# when the visible-prefix backtest says a geology/PF candidate beats the default tracker.
import os as _gold_os
import glob as _gold_glob
import json as _gold_json
import time as _gold_time
import hashlib as _gold_hashlib
from pathlib import Path as _GoldPath

import numpy as _gold_np
import pandas as _gold_pd

_GOLD_ENABLE = _gold_os.environ.get('ROGII_GOLD_PREFIX_CAL', '1') == '1'
_GOLD_PROFILE = _gold_os.environ.get('ROGII_GOLD_PROFILE', 'balanced').strip().lower()
_GOLD_INCLUDE_PF = _gold_os.environ.get('ROGII_GOLD_INCLUDE_PF', '1') == '1'
_GOLD_CAL_SEEDS = int(_gold_os.environ.get('ROGII_GOLD_CAL_SEEDS', '24'))
_GOLD_FINAL_SEEDS = int(_gold_os.environ.get('ROGII_GOLD_FINAL_SEEDS', '48'))
_GOLD_PARTICLES = int(_gold_os.environ.get('ROGII_GOLD_PARTICLES', '350'))
_GOLD_CUT_FRACS = tuple(float(x) for x in _gold_os.environ.get('ROGII_GOLD_CUT_FRACS', '0.55,0.70,0.84').split(',') if x.strip())
_GOLD_MAX_WELLS = int(_gold_os.environ.get('ROGII_GOLD_MAX_WELLS', '1000000'))

# Tuned hyper-parameter profiles (conservative/balanced/aggressive)
_GOLD_PROFILES = {
    'conservative': dict(min_gain=1.00, max_best=9.0, min_consistency=0.67, base=0.06, gain_scale=0.12, margin_scale=0.04, quality_bonus=0.02, cap=0.22, clip_base=8.0, clip_gain=3.0, clip_max=18.0, delta_soft=22.0, p95_hard=55.0),
    'balanced':     dict(min_gain=0.55, max_best=12.0, min_consistency=0.50, base=0.08, gain_scale=0.20, margin_scale=0.06, quality_bonus=0.04, cap=0.36, clip_base=10.0, clip_gain=4.5, clip_max=28.0, delta_soft=30.0, p95_hard=75.0),
    'aggressive':   dict(min_gain=0.25, max_best=15.0, min_consistency=0.34, base=0.12, gain_scale=0.32, margin_scale=0.10, quality_bonus=0.06, cap=0.56, clip_base=14.0, clip_gain=7.0, clip_max=45.0, delta_soft=42.0, p95_hard=110.0),
}
if _GOLD_PROFILE not in _GOLD_PROFILES:
    print(f'Unknown ROGII_GOLD_PROFILE={_GOLD_PROFILE!r}; using balanced')
    _GOLD_PROFILE = 'balanced'


# Resolve the working directory for Gold output
def _gold_work_dir():
    return _GoldPath('/kaggle/working') if _GoldPath('/kaggle/working').exists() else _GoldPath('.')


# Locate the competition data directory
def _gold_find_data():
    candidates = []
    obj = globals().get('CFG')
    if obj is not None:
        for attr in ('dataset_path', 'DATA'):
            if hasattr(obj, attr):
                candidates.append(_GoldPath(getattr(obj, attr)))
    candidates.extend([
        _GoldPath('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
        _GoldPath('/kaggle/input/rogii-wellbore-geology-prediction'),
        _GoldPath('.'),
    ])
    for c in candidates:
        try:
            if (c / 'train').exists() and (c / 'test').exists() and (c / 'sample_submission.csv').exists():
                return c
        except Exception:
            pass
    for p in _gold_glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True):
        c = _GoldPath(p).parent
        if (c / 'train').exists() and (c / 'test').exists():
            return c
    raise RuntimeError('Could not locate ROGII data directory')


def _gold_split_ids(df):
    out = df.copy()
    parts = out['id'].astype(str).str.rsplit('_', n=1, expand=True)
    if parts.shape[1] != 2:
        raise RuntimeError('Unexpected id format; expected well_rowindex')
    out['well'] = parts[0]
    out['row_idx'] = parts[1].astype(int)
    return out


def _gold_rmse(a, b):
    a = _gold_np.asarray(a, dtype=float)
    b = _gold_np.asarray(b, dtype=float)
    m = _gold_np.isfinite(a) & _gold_np.isfinite(b)
    if int(m.sum()) == 0:
        return float('inf')
    d = a[m] - b[m]
    return float(_gold_np.sqrt(_gold_np.mean(d * d)))


def _gold_sha256(path):
    h = _gold_hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def _gold_robust_poly_predict(x_known, y_known, x_all, deg):
    x_known = _gold_np.asarray(x_known, dtype=float)
    y_known = _gold_np.asarray(y_known, dtype=float)
    x_all = _gold_np.asarray(x_all, dtype=float)
    m = _gold_np.isfinite(x_known) & _gold_np.isfinite(y_known)
    x_known = x_known[m]
    y_known = y_known[m]
    if len(x_known) < 3:
        fill = float(_gold_np.nanmedian(y_known)) if len(y_known) else 0.0
        return _gold_np.full_like(x_all, fill, dtype=float)
    deg = int(min(max(1, deg), len(x_known) - 1))
    x0 = float(x_known[0])
    xs = float(_gold_np.nanmax(x_known) - _gold_np.nanmin(x_known))
    if (not _gold_np.isfinite(xs)) or xs < 1e-6:
        xs = 1.0
    xk = (x_known - x0) / xs
    xa = (x_all - x0) / xs
    try:
        coef = _gold_np.polyfit(xk, y_known, deg)
        for _ in range(5):
            fit = _gold_np.polyval(coef, xk)
            res = y_known - fit
            sc = 1.4826 * float(_gold_np.nanmedian(_gold_np.abs(res - _gold_np.nanmedian(res)))) + 1e-6
            weights = 1.0 / (1.0 + (res / (2.5 * sc)) ** 2)
            coef = _gold_np.polyfit(xk, y_known, deg, w=weights)
        return _gold_np.polyval(coef, xa).astype(float)
    except Exception:
        return _gold_np.full_like(x_all, float(_gold_np.nanmedian(y_known)), dtype=float)


def _gold_variant_grid():
    variants = set()
    try:
        variants.update(SELECTOR_BIN_VARIANTS.values())
        variants.add(SELECTOR_GLOBAL_VARIANT)
    except Exception:
        pass
    for scale in (3, 5, 8, 12):
        for hold in (0.0, 0.05, 0.10, 0.15, 0.20, 0.25):
            variants.add(f'pf_scale_{scale:g}_hold_{hold:g}')
        for beam in (0.05, 0.10, 0.20, 0.30):
            for hold in (0.0, 0.05, 0.10, 0.15, 0.20):
                variants.add(f'pf_scale_{scale:g}_beam_{beam:g}_hold_{hold:g}')
    return sorted(variants)


def _gold_tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
    tw_g = tw_tr.dropna(subset=['Geology']) if 'Geology' in tw_tr.columns else tw_tr.copy()
    if 'Geology' in tw_g.columns and ref_col in hw_tr.columns:
        ref_tvt = tw_g.loc[tw_g['Geology'] == ref_col, 'TVT'].min()
        if _gold_np.isnan(ref_tvt):
            ref_col = tw_g['Geology'].iloc[0]
            ref_tvt = tw_g.loc[tw_g['Geology'] == ref_col, 'TVT'].min()
        offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
        return (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset).to_numpy(dtype=float)
    return hw_tr['TVT'].to_numpy(dtype=float)


def _gold_contact_candidate(wid, hw, data_dir):
    out = {}
    try:
        hw_tr_path = data_dir / 'train' / f'{wid}__horizontal_well.csv'
        tw_tr_path = data_dir / 'train' / f'{wid}__typewell.csv'
        if not hw_tr_path.exists() or not tw_tr_path.exists():
            return out
        hw_tr = _gold_pd.read_csv(hw_tr_path)
        tw_tr = _gold_pd.read_csv(tw_tr_path)
        phys = _gold_tvt_from_contacts(hw_tr, tw_tr)
        md = hw_tr['MD'].to_numpy(dtype=float)
        m = _gold_np.isfinite(md) & _gold_np.isfinite(phys)
        if int(m.sum()) < 100:
            return out
        order = _gold_np.argsort(md[m])
        md_s = md[m][order]
        ph_s = phys[m][order]
        pred = _gold_np.interp(hw['MD'].to_numpy(dtype=float), md_s, ph_s, left=_gold_np.nan, right=_gold_np.nan)
        out['contact_md_lookup'] = pred.astype(float)
    except Exception:
        return out
    return out


def _gold_poly_candidates(hw_masked):
    out = {}
    tvt = hw_masked['TVT_input'].to_numpy(dtype=float)
    md = hw_masked['MD'].to_numpy(dtype=float)
    z = hw_masked['Z'].to_numpy(dtype=float)
    kn = _gold_np.flatnonzero(_gold_np.isfinite(tvt) & _gold_np.isfinite(md) & _gold_np.isfinite(z))
    if len(kn) < 30:
        return out
    u = tvt + z
    for tail in (80, 160, 320, 640, 1000000):
        sel = kn[-min(int(tail), len(kn)):]
        if len(sel) < 30:
            continue
        tag = 'all' if tail >= 1000000 else f'tail{tail}'
        for deg in (1, 2, 3):
            if len(sel) < deg + 12:
                continue
            uhat = _gold_robust_poly_predict(md[sel], u[sel], md, deg)
            out[f'poly_u_deg{deg}_{tag}'] = (uhat - z).astype(float)
    return out


def _gold_surface_candidates(hw_masked, wid, data_dir):
    out = {}
    tvt = hw_masked['TVT_input'].to_numpy(dtype=float)
    z = hw_masked['Z'].to_numpy(dtype=float)
    xy = hw_masked[['X', 'Y']].to_numpy(dtype=float)
    kn = _gold_np.isfinite(tvt) & _gold_np.isfinite(z) & _gold_np.isfinite(xy).all(axis=1)
    if int(kn.sum()) < 30:
        return out
    formations = list(globals().get('FORMATIONS', ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']))
    fi = globals().get('_FI', globals().get('FI', None))
    di = globals().get('_DI', globals().get('DI', None))
    surf_names = []
    try:
        if fi is not None:
            form_all, _ = fi.impute(xy, self_wid=None)
            form_all = _gold_np.asarray(form_all, dtype=float)
            for i, fn in enumerate(formations[:form_all.shape[1]]):
                f = form_all[:, i]
                good = kn & _gold_np.isfinite(f)
                if int(good.sum()) < 30:
                    continue
                b_med = float(_gold_np.nanmedian(tvt[good] + z[good] - f[good]))
                out[f'surface_{fn}_median'] = (-z + f + b_med).astype(float)
                surf_names.append(f'surface_{fn}_median')
                if callable(globals().get('seg_b_well')):
                    try:
                        b_full, _, _, b_late, b_wls = seg_b_well(
                            tvt[good].astype(_gold_np.float32),
                            z[good].astype(_gold_np.float32),
                            f[good].astype(_gold_np.float32),
                        )
                        out[f'surface_{fn}_full'] = (-z + f + float(b_full)).astype(float)
                        out[f'surface_{fn}_late'] = (-z + f + float(b_late)).astype(float)
                        out[f'surface_{fn}_wls'] = (-z + f + float(b_wls)).astype(float)
                        surf_names.extend([f'surface_{fn}_full', f'surface_{fn}_late', f'surface_{fn}_wls'])
                    except Exception:
                        pass
    except Exception as e:
        print('surface imputer skipped', wid, e)
    try:
        if di is not None:
            dense, _, _ = di.impute(xy, self_wid=None)
            dense = _gold_np.asarray(dense, dtype=float)
            good = kn & _gold_np.isfinite(dense)
            if int(good.sum()) >= 30:
                b_med = float(_gold_np.nanmedian(tvt[good] + z[good] - dense[good]))
                out['dense_ancc_median'] = (-z + dense + b_med).astype(float)
                surf_names.append('dense_ancc_median')
                if callable(globals().get('seg_b_well')):
                    try:
                        b_full, _, _, b_late, b_wls = seg_b_well(
                            tvt[good].astype(_gold_np.float32),
                            z[good].astype(_gold_np.float32),
                            dense[good].astype(_gold_np.float32),
                        )
                        out['dense_ancc_full'] = (-z + dense + float(b_full)).astype(float)
                        out['dense_ancc_late'] = (-z + dense + float(b_late)).astype(float)
                        out['dense_ancc_wls'] = (-z + dense + float(b_wls)).astype(float)
                        surf_names.extend(['dense_ancc_full', 'dense_ancc_late', 'dense_ancc_wls'])
                    except Exception:
                        pass
    except Exception as e:
        print('dense imputer skipped', wid, e)
    ens_names = [n for n in surf_names if n in out]
    if len(ens_names) >= 2:
        errs = _gold_np.array([_gold_rmse(out[n][kn], tvt[kn]) for n in ens_names], dtype=float)
        finite = _gold_np.isfinite(errs)
        if int(finite.sum()) >= 2:
            names = [n for n, ok in zip(ens_names, finite) if ok]
            errs = errs[finite]
            weights = 1.0 / _gold_np.maximum(errs, 0.25) ** 2
            weights = weights / weights.sum()
            mat = _gold_np.vstack([out[n] for n in names])
            out['surface_weighted_prefix'] = (weights[:, None] * mat).sum(axis=0).astype(float)
    out.update(_gold_contact_candidate(wid, hw_masked, data_dir))
    return out


def _gold_pf_candidates(hw_masked, tw, variants, n_seeds, n_particles):
    out = {}
    if not _GOLD_INCLUDE_PF:
        return out
    if not callable(globals().get('run_pf_lik_ensemble_scales')) or not callable(globals().get('apply_selector_variant')):
        return out
    kn = hw_masked[hw_masked['TVT_input'].notna()]
    ev = hw_masked[hw_masked['TVT_input'].isna()]
    if len(kn) < 30 or len(ev) == 0:
        return out
    try:
        pf_by_scale = run_pf_lik_ensemble_scales(
            hw_masked,
            tw,
            scales=tuple(globals().get('SELECTOR_SCALES', (3.0, 5.0, 8.0, 12.0))),
            n_particles=int(n_particles),
            n_seeds=int(n_seeds),
        )
        try:
            tvt_beam = run_beam_ensemble(hw_masked, tw)
        except Exception:
            tvt_beam = pf_by_scale.get('pf_mean')
            if tvt_beam is None:
                tvt_beam = next(iter(pf_by_scale.values()))
        last_known_tvt = float(kn['TVT_input'].iloc[-1])
        for variant in variants:
            try:
                pred = apply_selector_variant(variant, pf_by_scale, tvt_beam, last_known_tvt)
                if pred is not None and len(pred) == len(hw_masked):
                    out['pf|' + variant] = _gold_np.asarray(pred, dtype=float)
            except Exception:
                pass
    except Exception as e:
        print('PF calibration skipped:', e)
    return out


def _gold_candidate_pool(wid, hw_masked, tw, data_dir, variants, include_pf=True, n_seeds=24, n_particles=350):
    pool = {}
    pool.update(_gold_poly_candidates(hw_masked))
    pool.update(_gold_surface_candidates(hw_masked, wid, data_dir))
    if include_pf:
        pool.update(_gold_pf_candidates(hw_masked, tw, variants, n_seeds=n_seeds, n_particles=n_particles))
    clean = {}
    for name, pred in pool.items():
        arr = _gold_np.asarray(pred, dtype=float)
        if len(arr) == len(hw_masked) and _gold_np.isfinite(arr).sum() >= max(20, len(hw_masked) // 20):
            clean[name] = arr
    return clean


def _gold_default_pf_name(hw):
    try:
        return 'pf|' + selector_well_code(hw)[1]
    except Exception:
        try:
            return 'pf|' + SELECTOR_GLOBAL_VARIANT
        except Exception:
            return None


def _gold_calibrate_well(wid, hw, tw, data_dir, variants):
    tvt = hw['TVT_input'].to_numpy(dtype=float)
    is_known = _gold_np.isfinite(tvt)
    is_hidden = ~is_known
    if not bool(is_hidden.any()):
        return None
    first_hidden = int(_gold_np.flatnonzero(is_hidden)[0])
    known_prefix = _gold_np.flatnonzero(is_known & (_gold_np.arange(len(hw)) < first_hidden))
    if len(known_prefix) < 140:
        return dict(well=wid, status='skip_short_prefix', known_prefix=int(len(known_prefix)))
    cuts = []
    for frac in _GOLD_CUT_FRACS:
        cut_pos = int(round(len(known_prefix) * float(frac)))
        cut_pos = max(50, min(cut_pos, len(known_prefix) - 35))
        if cut_pos <= 0 or cut_pos >= len(known_prefix):
            continue
        cutoff_idx = int(known_prefix[cut_pos - 1])
        hold_idx = known_prefix[cut_pos:]
        if len(hold_idx) >= 35:
            cuts.append((float(frac), cutoff_idx, hold_idx))
    if not cuts:
        return dict(well=wid, status='skip_no_holdout', known_prefix=int(len(known_prefix)))
    scores = {}
    cut_rows = []
    default_name = _gold_default_pf_name(hw)
    for frac, cutoff_idx, hold_idx in cuts:
        hw_m = hw.copy(deep=True)
        hw_m.loc[hw_m.index > cutoff_idx, 'TVT_input'] = _gold_np.nan
        pool = _gold_candidate_pool(
            wid, hw_m, tw, data_dir, variants,
            include_pf=_GOLD_INCLUDE_PF,
            n_seeds=_GOLD_CAL_SEEDS,
            n_particles=_GOLD_PARTICLES,
        )
        y = tvt[hold_idx]
        row = {'well': wid, 'cut_frac': frac, 'holdout_rows': int(len(hold_idx)), 'candidates': int(len(pool))}
        local = []
        for name, pred in pool.items():
            err = _gold_rmse(pred[hold_idx], y)
            if _gold_np.isfinite(err):
                scores.setdefault(name, []).append(err)
                local.append((err, name))
        local.sort()
        if local:
            row['best_name'] = local[0][1]
            row['best_rmse'] = float(local[0][0])
            if default_name in pool:
                row['default_rmse'] = float(_gold_rmse(pool[default_name][hold_idx], y))
            else:
                row['default_rmse'] = float('nan')
        cut_rows.append(row)
    if not scores:
        return dict(well=wid, status='skip_no_scores', known_prefix=int(len(known_prefix)))
    agg = {}
    for name, vals in scores.items():
        vals = _gold_np.asarray(vals, dtype=float)
        agg[name] = float(_gold_np.nanmedian(vals) + 0.10 * _gold_np.nanstd(vals))
    ordered = sorted((v, k) for k, v in agg.items() if _gold_np.isfinite(v))
    if not ordered:
        return dict(well=wid, status='skip_nonfinite_scores', known_prefix=int(len(known_prefix)))
    best_score, best_name = ordered[0]
    second_score = ordered[1][0] if len(ordered) > 1 else best_score
    if default_name is not None and default_name in agg:
        default_score = float(agg[default_name])
    else:
        pf_scores = [v for k, v in agg.items() if k.startswith('pf|')]
        default_score = float(_gold_np.nanmedian(pf_scores)) if pf_scores else float(second_score)
    consistency = 0.0
    comparable = 0
    for row in cut_rows:
        if _gold_np.isfinite(row.get('default_rmse', _gold_np.nan)):
            comparable += 1
            if row.get('best_rmse', float('inf')) <= row['default_rmse'] - 0.25:
                consistency += 1.0
    if comparable:
        consistency /= comparable
    else:
        winners = [r.get('best_name') for r in cut_rows if r.get('best_name')]
        consistency = float(sum(w == best_name for w in winners)) / max(1, len(winners))
    return dict(
        well=wid,
        status='ok',
        known_prefix=int(len(known_prefix)),
        cuts=int(len(cut_rows)),
        candidate_count=int(len(agg)),
        best_name=best_name,
        best_score=float(best_score),
        second_score=float(second_score),
        default_name=default_name,
        default_score=float(default_score),
        gain=float(default_score - best_score),
        rank_margin=float(second_score - best_score),
        consistency=float(consistency),
        cut_rows=cut_rows,
    )


def _gold_alpha(report, delta_rmse, delta_p95, profile_name):
    p = _GOLD_PROFILES[profile_name]
    if report.get('status') != 'ok':
        return 0.0
    gain = float(report.get('gain', 0.0))
    best = float(report.get('best_score', float('inf')))
    margin = float(report.get('rank_margin', 0.0))
    consistency = float(report.get('consistency', 0.0))
    if (not _gold_np.isfinite(best)) or best > p['max_best'] or gain < p['min_gain'] or consistency < p['min_consistency']:
        return 0.0
    alpha = p['base']
    alpha += p['gain_scale'] * min(max(gain, 0.0), 5.0) / 5.0
    alpha += p['margin_scale'] * min(max(margin, 0.0), 3.0) / 3.0
    if best <= 5.0:
        alpha += p['quality_bonus']
    best_name = str(report.get('best_name', ''))
    if (best_name.startswith('surface_') or best_name.startswith('dense_') or best_name.startswith('poly_') or best_name.startswith('contact_')) and consistency >= 0.67:
        alpha += 0.03 if profile_name != 'aggressive' else 0.06
    if _gold_np.isfinite(delta_rmse) and delta_rmse > p['delta_soft']:
        alpha *= max(0.20, p['delta_soft'] / max(delta_rmse, 1e-6))
    if _gold_np.isfinite(delta_p95) and delta_p95 > p['p95_hard']:
        return 0.0
    return float(min(p['cap'], max(0.0, alpha)))


def _gold_profile_output(base_sub, candidate_by_id, reports_by_well, profile_name):
    prof = _GOLD_PROFILES[profile_name]
    out = base_sub.copy()
    move_rows = []
    for wid, rep in reports_by_well.items():
        ids = out.loc[out['well'] == wid, 'id'].astype(str).tolist()
        if not ids:
            continue
        cand = _gold_np.array([candidate_by_id.get(i, _gold_np.nan) for i in ids], dtype=float)
        idx = out.index[out['well'] == wid].to_numpy()
        base = out.loc[idx, 'tvt'].to_numpy(dtype=float)
        ok = _gold_np.isfinite(cand) & _gold_np.isfinite(base)
        if int(ok.sum()) != len(base):
            rep = dict(rep)
            rep['apply_status'] = 'skip_nonfinite_candidate'
            move_rows.append(rep)
            continue
        diff = cand - base
        delta_rmse = float(_gold_np.sqrt(_gold_np.mean(diff * diff))) if len(diff) else float('nan')
        delta_p95 = float(_gold_np.quantile(_gold_np.abs(diff), 0.95)) if len(diff) else float('nan')
        alpha = _gold_alpha(rep, delta_rmse, delta_p95, profile_name)
        gain = max(0.0, float(rep.get('gain', 0.0)))
        max_move = min(prof['clip_max'], prof['clip_base'] + prof['clip_gain'] * _gold_np.sqrt(gain + 1e-9))
        ramp = 1.0 - _gold_np.exp(-_gold_np.arange(len(diff), dtype=float) / max(80.0, 0.12 * max(1, len(diff))))
        move = _gold_np.clip(alpha * ramp * diff, -max_move, max_move)
        out.loc[idx, 'tvt'] = base + move
        row = dict(rep)
        row.update(dict(
            profile=profile_name,
            alpha=float(alpha),
            delta_rmse_vs_public=float(delta_rmse),
            delta_p95_vs_public=float(delta_p95),
            max_move_clip=float(max_move),
            applied_rows=int(len(idx)),
            mean_abs_move=float(_gold_np.mean(_gold_np.abs(move))) if len(move) else 0.0,
            max_abs_move=float(_gold_np.max(_gold_np.abs(move))) if len(move) else 0.0,
            apply_status='applied' if alpha > 0 else 'kept_public',
        ))
        move_rows.append(row)
    return out, move_rows


def _gold_reapply_guarded_contact_override(sub_df, data_dir):
    sub = _gold_split_ids(sub_df[['id', 'tvt']])
    pred = dict(zip(sub['id'].astype(str), sub['tvt'].astype(float)))
    train_wells = set(p.stem.replace('__horizontal_well', '') for p in (data_dir / 'train').glob('*__horizontal_well.csv'))
    n_ok = 0
    n_skip = 0
    for wid, g in sub.groupby('well'):
        if wid not in train_wells:
            continue
        try:
            hw_te = _gold_pd.read_csv(data_dir / 'test' / f'{wid}__horizontal_well.csv')
            hw_tr = _gold_pd.read_csv(data_dir / 'train' / f'{wid}__horizontal_well.csv')
            tw_tr = _gold_pd.read_csv(data_dir / 'train' / f'{wid}__typewell.csv')
            phys = _gold_tvt_from_contacts(hw_tr, tw_tr)
            md_raw = hw_tr['MD'].to_numpy(dtype=float)
            m = _gold_np.isfinite(phys) & _gold_np.isfinite(md_raw)
            if int(m.sum()) < 100:
                n_skip += 1
                continue
            order = _gold_np.argsort(md_raw[m])
            md_tr = md_raw[m][order]
            ph_tr = phys[m][order]
            kn = hw_te[hw_te['TVT_input'].notna()]
            kn = kn[(kn['MD'] >= md_tr[0]) & (kn['MD'] <= md_tr[-1])]
            if len(kn) < 50:
                n_skip += 1
                continue
            rk = _gold_rmse(_gold_np.interp(kn['MD'].to_numpy(dtype=float), md_tr, ph_tr), kn['TVT_input'].to_numpy(dtype=float))
            if (not _gold_np.isfinite(rk)) or rk > 1.0:
                n_skip += 1
                continue
            md_te = hw_te['MD'].to_numpy(dtype=float)
            n_row = 0
            for rid, ri in zip(g['id'].astype(str).values, g['row_idx'].values):
                ri = int(ri)
                if 0 <= ri < len(md_te):
                    mte = float(md_te[ri])
                    if md_tr[0] <= mte <= md_tr[-1]:
                        pred[rid] = float(_gold_np.interp(mte, md_tr, ph_tr))
                        n_row += 1
            print('gold contact override OK %s rmse=%.4f rows=%d/%d' % (wid, rk, n_row, len(g)))
            n_ok += 1
        except Exception as e:
            print('gold contact override fallback', wid, e)
            n_skip += 1
    sub['tvt'] = sub['id'].astype(str).map(pred).astype(float)
    print('gold contact override done: overridden=%d skipped=%d' % (n_ok, n_skip))
    return sub[['id', 'tvt']]


def _gold_validate_and_write(sub, sample, path):
    out = sub[['id', 'tvt']].copy()
    out['id'] = out['id'].astype(str)
    out['tvt'] = out['tvt'].astype(float)
    if list(out.columns) != ['id', 'tvt']:
        raise RuntimeError('bad output columns')
    if len(out) != len(sample):
        raise RuntimeError('bad output length')
    if not out['id'].equals(sample['id'].astype(str)):
        raise RuntimeError('id order mismatch')
    if not _gold_np.isfinite(out['tvt'].to_numpy(dtype=float)).all():
        raise RuntimeError('non-finite tvt in output')
    out.to_csv(path, index=False)
    return out


if not _GOLD_ENABLE:
    print('Gold visible-prefix calibration disabled; keeping public submission.csv')
else:
    _gold_t0 = _gold_time.time()
    _GOLD_WORK = _gold_work_dir()
    _GOLD_DATA = _gold_find_data()
    _gold_sample = _gold_pd.read_csv(_GOLD_DATA / 'sample_submission.csv')[['id']].copy()
    _gold_sample['id'] = _gold_sample['id'].astype(str)
    _gold_base = _gold_pd.read_csv(_GOLD_WORK / 'submission.csv')[['id', 'tvt']].copy()
    _gold_base['id'] = _gold_base['id'].astype(str)
    _gold_base['tvt'] = _gold_base['tvt'].astype(float)
    _gold_validate_and_write(_gold_base, _gold_sample, _GOLD_WORK / 'submission_public_self_verified.csv')
    _gold_base = _gold_split_ids(_gold_base)
    _gold_variants = _gold_variant_grid()
    print('Gold visible-prefix calibration:', dict(
        profile=_GOLD_PROFILE,
        include_pf=_GOLD_INCLUDE_PF,
        cal_seeds=_GOLD_CAL_SEEDS,
        final_seeds=_GOLD_FINAL_SEEDS,
        particles=_GOLD_PARTICLES,
        cut_fracs=_GOLD_CUT_FRACS,
        variants=len(_gold_variants),
    ))

    _gold_reports = []
    _gold_cut_reports = []
    _gold_candidate_by_id = {}
    _gold_wells = list(_gold_base['well'].drop_duplicates())[:_GOLD_MAX_WELLS]
    for _wi, _wid in enumerate(_gold_wells, 1):
        try:
            _hw_path = _GOLD_DATA / 'test' / f'{_wid}__horizontal_well.csv'
            _tw_path = _GOLD_DATA / 'test' / f'{_wid}__typewell.csv'
            if not _hw_path.exists() or not _tw_path.exists():
                _gold_reports.append(dict(well=_wid, status='skip_missing_files'))
                continue
            _hw = _gold_pd.read_csv(_hw_path)
            _tw = _gold_pd.read_csv(_tw_path)
            print('[gold %d/%d] calibrating %s' % (_wi, len(_gold_wells), _wid), flush=True)
            _rep = _gold_calibrate_well(_wid, _hw, _tw, _GOLD_DATA, _gold_variants)
            if _rep is None:
                _rep = dict(well=_wid, status='skip_none')
            _cut_rows = _rep.pop('cut_rows', []) if isinstance(_rep, dict) else []
            _gold_cut_reports.extend(_cut_rows)
            if _rep.get('status') == 'ok':
                _best_name = _rep['best_name']
                _need_pf_final = str(_best_name).startswith('pf|')
                _pool_final = _gold_candidate_pool(
                    _wid, _hw, _tw, _GOLD_DATA, _gold_variants,
                    include_pf=_need_pf_final,
                    n_seeds=_GOLD_FINAL_SEEDS,
                    n_particles=_GOLD_PARTICLES,
                )
                if _best_name not in _pool_final and _need_pf_final:
                    _pool_final = _gold_candidate_pool(
                        _wid, _hw, _tw, _GOLD_DATA, _gold_variants,
                        include_pf=False,
                        n_seeds=0,
                        n_particles=_GOLD_PARTICLES,
                    )
                if _best_name in _pool_final:
                    _g = _gold_base[_gold_base['well'] == _wid]
                    _arr = _pool_final[_best_name]
                    for _rid, _ri in zip(_g['id'].astype(str).values, _g['row_idx'].astype(int).values):
                        if 0 <= int(_ri) < len(_arr) and _gold_np.isfinite(_arr[int(_ri)]):
                            _gold_candidate_by_id[_rid] = float(_arr[int(_ri)])
                    _rep['final_candidate_available'] = True
                else:
                    _rep['final_candidate_available'] = False
                    _rep['status'] = 'skip_no_final_candidate'
            _gold_reports.append(_rep)
            print('  report:', {k: _rep.get(k) for k in ['status', 'best_name', 'best_score', 'default_score', 'gain', 'consistency']}, flush=True)
        except Exception as _e:
            print('gold calibration fallback', _wid, _e)
            _gold_reports.append(dict(well=_wid, status='error', error=str(_e)))

    _gold_report_df = _gold_pd.DataFrame(_gold_reports)
    _gold_report_df.to_csv(_GOLD_WORK / 'gold_prefix_calibration_report.csv', index=False)
    if _gold_cut_reports:
        _gold_pd.DataFrame(_gold_cut_reports).to_csv(_GOLD_WORK / 'gold_prefix_cut_report.csv', index=False)
    _reports_by_well = {r.get('well'): r for r in _gold_reports if isinstance(r, dict) and r.get('well')}

    _profile_summaries = {}
    for _profile_name in ('conservative', 'balanced', 'aggressive'):
        _profile_sub, _move_rows = _gold_profile_output(_gold_base, _gold_candidate_by_id, _reports_by_well, _profile_name)
        _profile_sub = _gold_reapply_guarded_contact_override(_profile_sub[['id', 'tvt']], _GOLD_DATA)
        _path = _GOLD_WORK / f'submission_gold_prefix_{_profile_name}.csv'
        _profile_sub = _gold_validate_and_write(_profile_sub, _gold_sample, _path)
        _move_df = _gold_pd.DataFrame(_move_rows)
        _move_df.to_csv(_GOLD_WORK / f'gold_prefix_moves_{_profile_name}.csv', index=False)
        _profile_summaries[_profile_name] = dict(
            file=str(_path),
            rows=int(len(_profile_sub)),
            sha256=_gold_sha256(_path),
            applied_wells=int((_move_df.get('apply_status') == 'applied').sum()) if 'apply_status' in _move_df else 0,
            mean_abs_move=float(_move_df['mean_abs_move'].mean()) if 'mean_abs_move' in _move_df and len(_move_df) else 0.0,
            max_abs_move=float(_move_df['max_abs_move'].max()) if 'max_abs_move' in _move_df and len(_move_df) else 0.0,
        )

    _chosen_path = _GOLD_WORK / f'submission_gold_prefix_{_GOLD_PROFILE}.csv'
    _chosen = _gold_pd.read_csv(_chosen_path)
    _chosen = _gold_validate_and_write(_chosen, _gold_sample, _GOLD_WORK / 'submission.csv')
    _audit = dict(
        selected_profile=_GOLD_PROFILE,
        selected_sha256=_gold_sha256(_GOLD_WORK / 'submission.csv'),
        public_anchor_sha256=_gold_sha256(_GOLD_WORK / 'submission_public_self_verified.csv'),
        elapsed_sec=float(_gold_time.time() - _gold_t0),
        wells=int(len(_gold_wells)),
        candidates_with_final_values=int(len(_gold_candidate_by_id)),
        profiles=_profile_summaries,
    )
    with open(_GOLD_WORK / 'gold_prefix_submission_audit.json', 'w', encoding='utf-8') as f:
        _gold_json.dump(_audit, f, indent=2, sort_keys=True)
    print('Gold visible-prefix selected submission.csv:', _audit, flush=True)